In [1]:
# =============================================================================
# CELL 1 — Imports and environment verification
#
# Kernel: CFB Model (ARM)  ~/miniforge3/envs/cfb_model_arm/bin/python
#
# model_08: combined refit + full posterior checks in a single notebook.
# Three prior changes from model_06 (only these three — everything else
# identical to model_06 Cell 2):
#   r_negbinom   : Gamma(16.0, 2.0) -> Gamma(4.0, 0.5)  [mean=8, std=4]
#   sigma_attack : HalfNormal(0.1)  -> HalfNormal(0.25)
#   sigma_defense: HalfNormal(0.1)  -> HalfNormal(0.25)
#
# Root causes addressed:
#   VMR gap: r posterior 12-18 implied VMR 2.3-3.0 vs observed 4.8-7.2.
#            Gamma(4.0, 0.5) allows r ~ 5-8 as data prefers.
#   Mean compression: sigma_attack=0.018, sigma_defense=0.061 too small.
#                     95/131 teams outside +/-2 pt threshold.
#                     HalfNormal(0.25) permits wider team random effects.
#
# BFMI fix: model_06 did not capture energy. This notebook uses
#   extra_fields=('energy',) in MCMC.run() and saves energy in pkl.
#
# DB connection opened here and held open for the full notebook.
# Do not close until artifact save cell is complete.
# =============================================================================

import numpy as np
import pandas as pd
import psycopg2
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value
from jax import random
from dataclasses import dataclass
from typing import Optional
import arviz as az
import matplotlib.pyplot as plt
import pickle
import time
import os

ARTIFACTS_DIR = "/Users/kevinjohnson/cfb-analytics/artifacts"

conn = psycopg2.connect(
    host='127.0.0.1', port=5455, dbname='postgres',
    user='postgres', password='postgres'
)
cur = conn.cursor()

print(f"NumPyro version : {numpyro.__version__}")
print(f"JAX version     : {jax.__version__}")
print(f"JAX backend     : {jax.default_backend()}")
print(f"pandas          : {pd.__version__}")
print(f"numpy           : {np.__version__}")
print(f"arviz           : {az.__version__}")
print()
print("DB connection established.")
print(f"ARTIFACTS_DIR   : {ARTIFACTS_DIR}")
assert os.path.isdir(ARTIFACTS_DIR), f"Artifacts directory not found: {ARTIFACTS_DIR}"
print("Cell 1 complete.")

NumPyro version : 0.21.0
JAX version     : 0.10.0
JAX backend     : cpu
pandas          : 3.0.2
numpy           : 2.4.4
arviz           : 0.23.4

DB connection established.
ARTIFACTS_DIR   : /Users/kevinjohnson/cfb-analytics/artifacts
Cell 1 complete.


In [2]:
# =============================================================================
# CELL 2 — Conference index maps, GameData, and model_cfb()
#
# Copied verbatim from model_06 Cell 2 with THREE prior changes only:
#   r_negbinom   : Gamma(16.0, 2.0) -> Gamma(4.0, 0.5)  [mean=8, std=4]
#   sigma_attack : HalfNormal(0.1)  -> HalfNormal(0.25)
#   sigma_defense: HalfNormal(0.1)  -> HalfNormal(0.25)
#
# Every other prior, every feature, every scope set, every index array,
# every mask, every likelihood term — identical to model_06 Cell 2.
# Do not modify anything else.
# =============================================================================

# --- Conference index maps ---
CONFERENCES = [
    "ACC",               # 0
    "American Athletic", # 1
    "Big 12",            # 2
    "Big Ten",           # 3
    "Conference USA",    # 4
    "Mid-American",      # 5
    "Mountain West",     # 6
    "Pac-12",            # 7
    "SEC",               # 8
    "Sun Belt",          # 9
]
N_CONFERENCES = len(CONFERENCES)
conf_to_idx   = {c: i for i, c in enumerate(CONFERENCES)}
SUN_BELT_IDX  = conf_to_idx["Sun Belt"]

# --- Conference-scope sets ---
SCOPE_LAST3_OFF_EPA      = {"ACC", "Mid-American", "SEC"}
SCOPE_LAST3_DEF_EPA      = {"American Athletic", "Big Ten", "Conference USA",
                             "Mid-American", "Pac-12", "Sun Belt"}
SCOPE_LAST3_PTS_SCORED   = {"ACC", "Big 12", "Big Ten", "Conference USA",
                             "Mid-American", "Mountain West"}
SCOPE_LAST3_PTS_ALLOWED  = {"American Athletic", "Big Ten", "Conference USA",
                             "Mountain West", "Pac-12", "Sun Belt"}
SCOPE_DAYS_SINCE         = {"American Athletic", "Big 12"}
SCOPE_CLOSE_PLAY_COUNT   = {"ACC", "American Athletic", "Big 12",
                             "Mid-American", "Pac-12", "Sun Belt"}
SCOPE_ELEVATION          = {"Mountain West", "Big 12"}


def build_conf_mask(team_conferences, scope_set):
    return jnp.array(
        [1.0 if c in scope_set else 0.0 for c in team_conferences],
        dtype=jnp.float32
    )


# --- GameData dataclass ---
@dataclass
class GameData:
    # Index arrays (int32)
    team_idx  : jnp.ndarray
    opp_idx   : jnp.ndarray
    conf_idx  : jnp.ndarray

    # Home indicator
    is_home   : jnp.ndarray

    # Observed scores
    points    : Optional[jnp.ndarray]

    # Prior seeds
    sp_rating          : jnp.ndarray
    recruiting_3yr_avg : jnp.ndarray

    # Universal game-level features
    close_game_epa        : jnp.ndarray
    close_game_def_epa    : jnp.ndarray
    pregame_elo           : jnp.ndarray
    elo_sp_divergence     : jnp.ndarray
    last3_win_pct         : jnp.ndarray
    away_travel_distance  : jnp.ndarray
    away_tz_delta         : jnp.ndarray
    wind_chill            : jnp.ndarray
    rush_rate_std_downs   : jnp.ndarray
    rush_rate_pass_downs  : jnp.ndarray
    off_sack_rate_allowed : jnp.ndarray
    def_sack_rate         : jnp.ndarray

    # Archetype index arrays — int32, values 0-3
    off_archetype_idx : jnp.ndarray
    def_archetype_idx : jnp.ndarray

    # Conference-scoped features + masks (7 pairs)
    last3_off_epa_avg      : jnp.ndarray
    mask_last3_off_epa     : jnp.ndarray
    last3_def_epa_avg      : jnp.ndarray
    mask_last3_def_epa     : jnp.ndarray
    last3_pts_scored_avg   : jnp.ndarray
    mask_last3_pts_scored  : jnp.ndarray
    last3_pts_allowed_avg  : jnp.ndarray
    mask_last3_pts_allowed : jnp.ndarray
    days_since_last_game   : jnp.ndarray
    mask_days_since        : jnp.ndarray
    close_play_count_delta : jnp.ndarray
    mask_close_play_count  : jnp.ndarray
    away_elevation_delta   : jnp.ndarray
    mask_elevation         : jnp.ndarray


# --- model_cfb() ---
def model_cfb(data: GameData, N_teams: int):

    # ── League level ──────────────────────────────────────────────────────────
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))

    # CHANGED: Gamma(16.0, 2.0) -> Gamma(4.0, 0.5)  [mean=8, std=4]
    # Rationale: model_07 found r posterior 12-18 implies VMR 2.3-3.0;
    # observed VMR 4.8-7.2. Data prefers r ~ 5-8. Gamma(4,0.5) allows this.
    r_negbinom = numpyro.sample(
        "r_negbinom",
        dist.Gamma(4.0, 0.5),
        sample_shape=(N_CONFERENCES,)
    )

    # ── Conference level ──────────────────────────────────────────────────────
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(0.1))
    mu_conference    = numpyro.sample(
        "mu_conference",
        dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )

    # ── Team level — non-centered ─────────────────────────────────────────────
    # CHANGED: HalfNormal(0.1) -> HalfNormal(0.25)
    # Rationale: model_07 sigma_attack posterior mean 0.018 compressed team
    # effects too tightly. 95/131 teams outside +/-2 pt mean threshold.
    sigma_attack   = numpyro.sample("sigma_attack",   dist.HalfNormal(0.25))
    alpha_team_raw = numpyro.sample(
        "alpha_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    alpha_team = numpyro.deterministic("alpha_team", alpha_team_raw * sigma_attack)

    # CHANGED: HalfNormal(0.1) -> HalfNormal(0.25)
    sigma_defense  = numpyro.sample("sigma_defense",  dist.HalfNormal(0.25))
    delta_team_raw = numpyro.sample(
        "delta_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    delta_team = numpyro.deterministic("delta_team", delta_team_raw * sigma_defense)

    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team_raw   = numpyro.sample(
        "hfa_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    hfa_team = numpyro.deterministic("hfa_team", hfa_team_raw * sigma_hfa_team)

    # ── Prior seeds ───────────────────────────────────────────────────────────
    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 0.15))

    rec_weight_other   = numpyro.sample(
        "rec_weight_other", dist.Normal(0.0, 0.5), sample_shape=(N_CONFERENCES - 1,)
    )
    rec_weight_sunbelt = numpyro.sample(
        "rec_weight_sunbelt", dist.TruncatedNormal(0.0, 0.5, high=0.0)
    )
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])

    # ── Game-level coefficients ───────────────────────────────────────────────
    b_close_game_epa      = numpyro.sample("b_close_game_epa",      dist.Normal(0.0, 0.10))
    b_close_game_def_epa  = numpyro.sample("b_close_game_def_epa",  dist.Normal(0.0, 0.10))
    b_pregame_elo         = numpyro.sample("b_pregame_elo",         dist.Normal(0.0, 0.15))
    b_elo_sp_divergence   = numpyro.sample("b_elo_sp_divergence",   dist.Normal(0.0, 0.15))
    b_last3_win_pct       = numpyro.sample("b_last3_win_pct",       dist.Normal(0.0, 0.15))
    b_away_travel_distance= numpyro.sample("b_away_travel_distance",dist.Normal(0.0, 0.15))
    b_away_tz_delta       = numpyro.sample("b_away_tz_delta",       dist.Normal(0.0, 0.15))
    b_wind_chill          = numpyro.sample("b_wind_chill",          dist.Normal(0.0, 0.15))
    b_rush_rate_std_downs = numpyro.sample("b_rush_rate_std_downs", dist.Normal(0.0, 0.15))
    b_rush_rate_pass_downs= numpyro.sample("b_rush_rate_pass_downs",dist.Normal(0.0, 0.15))
    b_off_sack_rate_allowed=numpyro.sample("b_off_sack_rate_allowed",dist.Normal(0.0, 0.15))
    b_def_sack_rate       = numpyro.sample("b_def_sack_rate",       dist.Normal(0.0, 0.15))

    b_off_archetype = numpyro.sample(
        "b_off_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_def_archetype = numpyro.sample(
        "b_def_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )

    b_last3_off_epa_avg      = numpyro.sample("b_last3_off_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_def_epa_avg      = numpyro.sample("b_last3_def_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_pts_scored_avg   = numpyro.sample("b_last3_pts_scored_avg",   dist.Normal(0.0, 0.15))
    b_last3_pts_allowed_avg  = numpyro.sample("b_last3_pts_allowed_avg",  dist.Normal(0.0, 0.15))
    b_days_since_last_game   = numpyro.sample("b_days_since_last_game",   dist.Normal(0.0, 0.15))
    b_close_play_count_delta = numpyro.sample("b_close_play_count_delta", dist.Normal(0.0, 0.15))
    b_away_elevation_delta   = numpyro.sample("b_away_elevation_delta",   dist.Normal(0.0, 0.15))

    # ── Linear predictor ──────────────────────────────────────────────────────
    log_mu = (
        mu_league
        + mu_conference[data.conf_idx]
        + alpha_team[data.team_idx]
        - delta_team[data.opp_idx]
        + (hfa_league + hfa_team[data.team_idx]) * data.is_home
        + sp_weight           * data.sp_rating
        + rec_weight[data.conf_idx] * data.recruiting_3yr_avg
        + b_close_game_epa      * data.close_game_epa
        + b_close_game_def_epa  * data.close_game_def_epa
        + b_pregame_elo         * data.pregame_elo
        + b_elo_sp_divergence   * data.elo_sp_divergence
        + b_last3_win_pct       * data.last3_win_pct
        + b_away_travel_distance * data.away_travel_distance
        + b_away_tz_delta        * data.away_tz_delta
        + b_wind_chill           * data.wind_chill
        + b_rush_rate_std_downs  * data.rush_rate_std_downs
        + b_rush_rate_pass_downs * data.rush_rate_pass_downs
        + b_off_sack_rate_allowed * data.off_sack_rate_allowed
        + b_def_sack_rate         * data.def_sack_rate
        + b_off_archetype[data.off_archetype_idx]
        + b_def_archetype[data.def_archetype_idx]
        + b_last3_off_epa_avg     * data.mask_last3_off_epa    * data.last3_off_epa_avg
        + b_last3_def_epa_avg     * data.mask_last3_def_epa    * data.last3_def_epa_avg
        + b_last3_pts_scored_avg  * data.mask_last3_pts_scored * data.last3_pts_scored_avg
        + b_last3_pts_allowed_avg * data.mask_last3_pts_allowed * data.last3_pts_allowed_avg
        + b_days_since_last_game  * data.mask_days_since        * data.days_since_last_game
        + b_close_play_count_delta * data.mask_close_play_count * data.close_play_count_delta
        + b_away_elevation_delta  * data.mask_elevation         * data.away_elevation_delta
    )

    # ── Likelihood ────────────────────────────────────────────────────────────
    mu = jnp.exp(log_mu).clip(1e-6)
    r  = r_negbinom[data.conf_idx].clip(1e-6)

    numpyro.sample(
        "obs",
        dist.NegativeBinomial2(mu, r, validate_args=False),
        obs=data.points
    )


print("Conference index map:")
for name, idx in conf_to_idx.items():
    tag = "  <- Sun Belt (rec_weight non-positive)" if idx == SUN_BELT_IDX else ""
    print(f"  {idx:2d} : {name}{tag}")
print(f"\nN_CONFERENCES : {N_CONFERENCES}")
print(f"SUN_BELT_IDX  : {SUN_BELT_IDX}")
print()
print("build_conf_mask() defined.")
print("GameData dataclass defined.")
print("model_cfb() defined.")
print("  r_negbinom       : Gamma(4.0, 0.5) x N_CONFERENCES  [mean=8, std=4]  CHANGED")
print("  sigma_conference : HalfNormal(0.1)")
print("  sigma_attack     : HalfNormal(0.25)                                   CHANGED")
print("  sigma_defense    : HalfNormal(0.25)                                   CHANGED")
print("  sigma_hfa_team   : HalfNormal(0.1)")
print("  sp_weight        : Normal(0.0, 0.15)")
print("  b_off_archetype  : Normal(0.0, 0.15) x 4")
print("  b_def_archetype  : Normal(0.0, 0.15) x 4")
print("  likelihood       : NegativeBinomial2(mu, r_negbinom[conf_idx])")
print("Cell 2 complete.")

Conference index map:
   0 : ACC
   1 : American Athletic
   2 : Big 12
   3 : Big Ten
   4 : Conference USA
   5 : Mid-American
   6 : Mountain West
   7 : Pac-12
   8 : SEC
   9 : Sun Belt  <- Sun Belt (rec_weight non-positive)

N_CONFERENCES : 10
SUN_BELT_IDX  : 9

build_conf_mask() defined.
GameData dataclass defined.
model_cfb() defined.
  r_negbinom       : Gamma(4.0, 0.5) x N_CONFERENCES  [mean=8, std=4]  CHANGED
  sigma_conference : HalfNormal(0.1)
  sigma_attack     : HalfNormal(0.25)                                   CHANGED
  sigma_defense    : HalfNormal(0.25)                                   CHANGED
  sigma_hfa_team   : HalfNormal(0.1)
  sp_weight        : Normal(0.0, 0.15)
  b_off_archetype  : Normal(0.0, 0.15) x 4
  b_def_archetype  : Normal(0.0, 0.15) x 4
  likelihood       : NegativeBinomial2(mu, r_negbinom[conf_idx])
Cell 2 complete.


In [3]:
# =============================================================================
# CELL 3 — Load training data from int.int_game_model_features
#
# Copied verbatim from model_06 Cell 3.
# Season filter: IN (2022, 2023, 2024). 2025 is holdout — never touched.
# FBS integrity check mandatory after load.
# All Decimal columns cast to float64 immediately.
# Expected: 3,214 rows, 1,607 games, 131 teams, 0 nulls.
# =============================================================================

load_sql = """
    SELECT *
    FROM int.int_game_model_features
    WHERE season IN (2022, 2023, 2024)
    ORDER BY season, game_id, team_name
"""

cur.execute(load_sql)
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
df = pd.DataFrame(rows, columns=cols)

numeric_cols = [c for c in df.columns if c not in
                ['team_name', 'opponent', 'conference']]
df[numeric_cols] = df[numeric_cols].astype(float)

# --- FBS integrity check ---
print("Conference distribution:")
print(df['conference'].value_counts().sort_index().to_string())
print()

assert 'FBS Independents' not in df['conference'].values, \
    "FBS Independents found in conference — fix before proceeding"
assert 2025 not in df['season'].values, \
    "2025 holdout found in training data — fix before proceeding"
assert df['game_id'].isna().sum() == 0, \
    "Null game_id found"

# --- Shape and basic sanity ---
print(f"Rows loaded        : {len(df):,}")
print(f"Columns            : {len(df.columns)}")
print(f"Unique games       : {df['game_id'].nunique():,}")
print(f"Unique teams       : {df['team_name'].nunique():,}")
print(f"Seasons            : {sorted(df['season'].unique().tolist())}")
print(f"Nulls anywhere     : {df.isna().sum().sum()}")
print(f"points_scored mean : {df['points_scored'].mean():.1f}")
print()

assert len(df) == 3214, \
    f"Expected 3,214 rows, got {len(df):,}"
assert df['game_id'].nunique() == 1607, \
    f"Expected 1,607 games, got {df['game_id'].nunique():,}"
assert df.isna().sum().sum() == 0, "Nulls found in training data"

print("All load checks passed.")
print("Cell 3 complete.")

Conference distribution:
conference
ACC                  364
American Athletic    320
Big 12               366
Big Ten              420
Conference USA       246
Mid-American         294
Mountain West        282
Pac-12               222
SEC                  358
Sun Belt             342

Rows loaded        : 3,214
Columns            : 31
Unique games       : 1,607
Unique teams       : 131
Seasons            : [2022.0, 2023.0, 2024.0]
Nulls anywhere     : 0
points_scored mean : 26.9

All load checks passed.
Cell 3 complete.


In [4]:
# =============================================================================
# CELL 4 — Build index arrays and conference masks
#
# Copied verbatim from model_06 Cell 4.
# team_idx and opp_idx: 0-based over 131 unique teams, pooled across seasons.
# conf_idx: maps conference string to conf_to_idx integer from Cell 2.
# All masks built from team_conferences list (row-level conference strings).
# =============================================================================

# --- Team index map ---
unique_teams = sorted(df['team_name'].unique())
N_teams      = len(unique_teams)
team_to_idx  = {name: i for i, name in enumerate(unique_teams)}

print(f"N_teams : {N_teams}")

# --- Verify all opponents are in the team map ---
unknown_opps = set(df['opponent'].unique()) - set(team_to_idx.keys())
assert len(unknown_opps) == 0, \
    f"Opponents not in team_to_idx: {unknown_opps}"

# --- Build index arrays ---
team_idx = jnp.array(df['team_name'].map(team_to_idx).values, dtype=jnp.int32)
opp_idx  = jnp.array(df['opponent'].map(team_to_idx).values,  dtype=jnp.int32)
conf_idx = jnp.array(df['conference'].map(conf_to_idx).values, dtype=jnp.int32)

# --- Verify conference mapping was complete ---
assert df['conference'].map(conf_to_idx).isna().sum() == 0, \
    f"Unmapped conferences: {set(df['conference'].unique()) - set(conf_to_idx.keys())}"

# --- Range assertions ---
assert int(team_idx.min()) >= 0 and int(team_idx.max()) < N_teams, \
    f"team_idx out of range: [{int(team_idx.min())}, {int(team_idx.max())}]"
assert int(opp_idx.min()) >= 0 and int(opp_idx.max()) < N_teams, \
    f"opp_idx out of range: [{int(opp_idx.min())}, {int(opp_idx.max())}]"
assert int(conf_idx.min()) >= 0 and int(conf_idx.max()) < N_CONFERENCES, \
    f"conf_idx out of range: [{int(conf_idx.min())}, {int(conf_idx.max())}]"

print(f"team_idx : shape={team_idx.shape}  min={int(team_idx.min())}  max={int(team_idx.max())}")
print(f"opp_idx  : shape={opp_idx.shape}  min={int(opp_idx.min())}  max={int(opp_idx.max())}")
print(f"conf_idx : shape={conf_idx.shape}  min={int(conf_idx.min())}  max={int(conf_idx.max())}")

# --- Build conference masks ---
team_conferences = df['conference'].tolist()

mask_last3_off_epa     = build_conf_mask(team_conferences, SCOPE_LAST3_OFF_EPA)
mask_last3_def_epa     = build_conf_mask(team_conferences, SCOPE_LAST3_DEF_EPA)
mask_last3_pts_scored  = build_conf_mask(team_conferences, SCOPE_LAST3_PTS_SCORED)
mask_last3_pts_allowed = build_conf_mask(team_conferences, SCOPE_LAST3_PTS_ALLOWED)
mask_days_since        = build_conf_mask(team_conferences, SCOPE_DAYS_SINCE)
mask_close_play_count  = build_conf_mask(team_conferences, SCOPE_CLOSE_PLAY_COUNT)
mask_elevation         = build_conf_mask(team_conferences, SCOPE_ELEVATION)

print()
print("Conference mask non-zero counts (of 3,214 rows):")
print(f"  mask_last3_off_epa     : {int(mask_last3_off_epa.sum()):,}")
print(f"  mask_last3_def_epa     : {int(mask_last3_def_epa.sum()):,}")
print(f"  mask_last3_pts_scored  : {int(mask_last3_pts_scored.sum()):,}")
print(f"  mask_last3_pts_allowed : {int(mask_last3_pts_allowed.sum()):,}")
print(f"  mask_days_since        : {int(mask_days_since.sum()):,}")
print(f"  mask_close_play_count  : {int(mask_close_play_count.sum()):,}")
print(f"  mask_elevation         : {int(mask_elevation.sum()):,}")

assert int(mask_last3_off_epa.sum()) == 364 + 294 + 358, \
    f"mask_last3_off_epa sum wrong: {int(mask_last3_off_epa.sum())}"

print()
print("All index and mask checks passed.")
print("Cell 4 complete.")

N_teams : 131
team_idx : shape=(3214,)  min=0  max=130
opp_idx  : shape=(3214,)  min=0  max=130
conf_idx : shape=(3214,)  min=0  max=9

Conference mask non-zero counts (of 3,214 rows):
  mask_last3_off_epa     : 1,016
  mask_last3_def_epa     : 1,844
  mask_last3_pts_scored  : 1,972
  mask_last3_pts_allowed : 1,832
  mask_days_since        : 686
  mask_close_play_count  : 1,908
  mask_elevation         : 648

All index and mask checks passed.
Cell 4 complete.


In [5]:
# =============================================================================
# CELL 5 — Construct GameData
#
# Copied verbatim from model_06 Cell 5.
# Data loaded from int.int_game_model_features is already fully preprocessed
# by model_03 Cell 7. DO NOT re-standardize anything here.
#
# Column name reference (exact names as they exist in the table):
#   close_game_epa_per_play       -> close_game_epa
#   close_game_def_epa_per_play   -> close_game_def_epa
#   away_travel_distance_mi       -> away_travel_distance
#   away_tz_delta_hrs             -> away_tz_delta
#   rush_rate_std_downs_delta     -> rush_rate_std_downs
#   rush_rate_pass_downs_delta    -> rush_rate_pass_downs
#   off_sack_rate_allowed_delta   -> off_sack_rate_allowed
#   def_sack_rate_delta           -> def_sack_rate
#   last3_points_scored_avg       -> last3_pts_scored_avg
#   last3_points_allowed_avg      -> last3_pts_allowed_avg
#   close_game_play_count_delta   -> close_play_count_delta
#   away_elevation_delta_ft       -> away_elevation_delta
# =============================================================================

def to_f32(col):
    return jnp.array(df[col].values, dtype=jnp.float32)

def to_i32(col):
    return jnp.array(df[col].values, dtype=jnp.int32)

training_data = GameData(
    team_idx  = team_idx,
    opp_idx   = opp_idx,
    conf_idx  = conf_idx,

    is_home   = jnp.array(df['is_home'].values, dtype=jnp.float32),
    points    = jnp.array(df['points_scored'].values, dtype=jnp.int32),

    sp_rating          = to_f32('sp_rating'),
    recruiting_3yr_avg = to_f32('recruiting_3yr_avg'),

    close_game_epa        = to_f32('close_game_epa_per_play'),
    close_game_def_epa    = to_f32('close_game_def_epa_per_play'),
    pregame_elo           = to_f32('pregame_elo'),
    elo_sp_divergence     = to_f32('elo_sp_divergence'),
    last3_win_pct         = to_f32('last3_win_pct'),
    away_travel_distance  = to_f32('away_travel_distance_mi'),
    away_tz_delta         = to_f32('away_tz_delta_hrs'),
    wind_chill            = to_f32('wind_chill'),
    rush_rate_std_downs   = to_f32('rush_rate_std_downs_delta'),
    rush_rate_pass_downs  = to_f32('rush_rate_pass_downs_delta'),
    off_sack_rate_allowed = to_f32('off_sack_rate_allowed_delta'),
    def_sack_rate         = to_f32('def_sack_rate_delta'),

    off_archetype_idx = to_i32('off_archetype_idx'),
    def_archetype_idx = to_i32('def_archetype_idx'),

    last3_off_epa_avg      = to_f32('last3_off_epa_avg'),
    mask_last3_off_epa     = mask_last3_off_epa,
    last3_def_epa_avg      = to_f32('last3_def_epa_avg'),
    mask_last3_def_epa     = mask_last3_def_epa,
    last3_pts_scored_avg   = to_f32('last3_points_scored_avg'),
    mask_last3_pts_scored  = mask_last3_pts_scored,
    last3_pts_allowed_avg  = to_f32('last3_points_allowed_avg'),
    mask_last3_pts_allowed = mask_last3_pts_allowed,
    days_since_last_game   = to_f32('days_since_last_game'),
    mask_days_since        = mask_days_since,
    close_play_count_delta = to_f32('close_game_play_count_delta'),
    mask_close_play_count  = mask_close_play_count,
    away_elevation_delta   = to_f32('away_elevation_delta_ft'),
    mask_elevation         = mask_elevation,
)

# --- Verification ---
import dataclasses

N_rows = len(df)
for field in dataclasses.fields(training_data):
    val = getattr(training_data, field.name)
    if val is not None:
        assert val.shape[0] == N_rows, \
            f"{field.name} shape[0] = {val.shape[0]}, expected {N_rows}"

print(f"All {len(dataclasses.fields(training_data))} GameData fields verified: shape[0] == {N_rows}.")
print()

# Confirm archetype index dtypes and ranges
assert training_data.off_archetype_idx.dtype == jnp.int32, \
    f"off_archetype_idx dtype wrong: {training_data.off_archetype_idx.dtype}"
assert training_data.def_archetype_idx.dtype == jnp.int32, \
    f"def_archetype_idx dtype wrong: {training_data.def_archetype_idx.dtype}"
assert int(training_data.off_archetype_idx.min()) >= 0 and \
       int(training_data.off_archetype_idx.max()) <= 3, \
    f"off_archetype_idx out of range: [{int(training_data.off_archetype_idx.min())}, {int(training_data.off_archetype_idx.max())}]"
assert int(training_data.def_archetype_idx.min()) >= 0 and \
       int(training_data.def_archetype_idx.max()) <= 3, \
    f"def_archetype_idx out of range: [{int(training_data.def_archetype_idx.min())}, {int(training_data.def_archetype_idx.max())}]"

print(f"off_archetype_idx : dtype={training_data.off_archetype_idx.dtype}  "
      f"range=[{int(training_data.off_archetype_idx.min())}, {int(training_data.off_archetype_idx.max())}]")
print(f"def_archetype_idx : dtype={training_data.def_archetype_idx.dtype}  "
      f"range=[{int(training_data.def_archetype_idx.min())}, {int(training_data.def_archetype_idx.max())}]")
print()

# Confirm key continuous features are on unit scale (model_03 already scaled)
print("Scale check on continuous features (expect mean≈0, std≈1):")
check_cols = [
    ('sp_rating',               training_data.sp_rating),
    ('close_game_epa',          training_data.close_game_epa),
    ('pregame_elo',             training_data.pregame_elo),
    ('rush_rate_std_downs',     training_data.rush_rate_std_downs),
    ('last3_win_pct',           training_data.last3_win_pct),
]
for name, arr in check_cols:
    mean = float(arr.mean())
    std  = float(arr.std())
    print(f"  {name:<30s} mean={mean:+.4f}  std={std:.4f}")

print()
print("Cell 5 complete.")

All 35 GameData fields verified: shape[0] == 3214.

off_archetype_idx : dtype=int32  range=[0, 3]
def_archetype_idx : dtype=int32  range=[0, 3]

Scale check on continuous features (expect mean≈0, std≈1):
  sp_rating                      mean=-0.0000  std=0.9998
  close_game_epa                 mean=-0.0016  std=0.9914
  pregame_elo                    mean=-0.0000  std=0.9998
  rush_rate_std_downs            mean=-0.0000  std=0.9937
  last3_win_pct                  mean=-0.0000  std=0.9998

Cell 5 complete.


In [6]:
# =============================================================================
# CELL 6 (REWRITE) — Run NUTS sampler (model_08 refit)
#
# Identical to previous Cell 6 except:
#   num_warmup : 1000 -> 2000
#
# Reason: sigma_attack R-hat = 1.0108 (threshold < 1.01) with ESS_bulk = 600.
# sigma_attack posterior mean ~0.024 sits near the HalfNormal boundary.
# Chain mixing is slow — longer warmup gives NUTS more adaptation steps.
# All other parameters passed. No prior changes. No other modifications.
#
# Expected wall-clock: ~20-25 min (5000 total iterations per chain vs 4000).
# =============================================================================

init_values = {
    "mu_league"          : jnp.array(3.3),
    "hfa_league"         : jnp.array(0.1),
    "r_negbinom"         : jnp.ones(N_CONFERENCES) * 8.0,
    "sigma_conference"   : jnp.array(0.05),
    "mu_conference"      : jnp.zeros(N_CONFERENCES),
    "sigma_attack"       : jnp.array(0.05),
    "alpha_team_raw"     : jnp.zeros(N_teams),
    "sigma_defense"      : jnp.array(0.05),
    "delta_team_raw"     : jnp.zeros(N_teams),
    "sigma_hfa_team"     : jnp.array(0.05),
    "hfa_team_raw"       : jnp.zeros(N_teams),
    "sp_weight"          : jnp.array(0.0),
    "rec_weight_other"   : jnp.zeros(N_CONFERENCES - 1),
    "rec_weight_sunbelt" : jnp.array(-0.1),
    "b_close_game_epa"        : jnp.array(0.0),
    "b_close_game_def_epa"    : jnp.array(0.0),
    "b_pregame_elo"           : jnp.array(0.0),
    "b_elo_sp_divergence"     : jnp.array(0.0),
    "b_last3_win_pct"         : jnp.array(0.0),
    "b_away_travel_distance"  : jnp.array(0.0),
    "b_away_tz_delta"         : jnp.array(0.0),
    "b_wind_chill"            : jnp.array(0.0),
    "b_rush_rate_std_downs"   : jnp.array(0.0),
    "b_rush_rate_pass_downs"  : jnp.array(0.0),
    "b_off_sack_rate_allowed" : jnp.array(0.0),
    "b_def_sack_rate"         : jnp.array(0.0),
    "b_off_archetype"         : jnp.zeros(4),
    "b_def_archetype"         : jnp.zeros(4),
    "b_last3_off_epa_avg"     : jnp.array(0.0),
    "b_last3_def_epa_avg"     : jnp.array(0.0),
    "b_last3_pts_scored_avg"  : jnp.array(0.0),
    "b_last3_pts_allowed_avg" : jnp.array(0.0),
    "b_days_since_last_game"  : jnp.array(0.0),
    "b_close_play_count_delta": jnp.array(0.0),
    "b_away_elevation_delta"  : jnp.array(0.0),
}

nuts_kernel = NUTS(
    model_cfb,
    target_accept_prob=0.9,
    init_strategy=init_to_value(values=init_values),
)

mcmc = MCMC(
    nuts_kernel,
    num_warmup=2000,
    num_samples=1000,
    num_chains=4,
    progress_bar=True,
)

print("Starting NUTS sampler (model_08 refit — 4 chains, 2000 warmup, 1000 samples)...")
print(f"  N_teams       : {N_teams}")
print(f"  N_games       : {len(df):,}")
print(f"  N_CONFERENCES : {N_CONFERENCES}")
print(f"  Prior changes : r_negbinom Gamma(4.0,0.5), sigma_attack/defense HalfNormal(0.25)")
print(f"  Warmup change : 1000 -> 2000  [fix for sigma_attack R-hat=1.0108]")
print()

numpyro.enable_validation(False)
try:
    t0 = time.time()
    mcmc.run(random.PRNGKey(42), data=training_data, N_teams=N_teams,
             extra_fields=('energy',))
    fit_time = time.time() - t0
finally:
    numpyro.enable_validation(True)

print(f"\nFit complete. Wall-clock time : {fit_time:.1f}s")

# --- Divergences ---
extra         = mcmc.get_extra_fields()
n_divergences = int(extra['diverging'].sum()) if 'diverging' in extra else 0
print(f"Divergences   : {n_divergences} / 4000")
assert n_divergences == 0, f"STOP — {n_divergences} divergences. Do not proceed."

# --- Energy (for BFMI) ---
energy = extra['energy']
print(f"Energy array  : shape={energy.shape}  (chains x samples for BFMI)")

# --- Samples ---
samples = mcmc.get_samples()

# r_negbinom per-conference summary
r_vals = samples['r_negbinom']
print(f"\nr_negbinom shape : {r_vals.shape}  (expect (4000, {N_CONFERENCES}))")
print("r_negbinom posterior means by conference:")
for i, conf in enumerate(CONFERENCES):
    print(f"  {conf:<22s} mean={float(r_vals[:, i].mean()):.4f}  "
          f"std={float(r_vals[:, i].std()):.4f}")

print(f"\nmu_league     mean={float(samples['mu_league'].mean()):.4f}")
print(f"hfa_league    mean={float(samples['hfa_league'].mean()):.4f}")
print(f"sp_weight     mean={float(samples['sp_weight'].mean()):.4f}")
print(f"sigma_attack  mean={float(samples['sigma_attack'].mean()):.4f}")
print(f"sigma_defense mean={float(samples['sigma_defense'].mean()):.4f}")

print(f"\nb_off_archetype shape : {samples['b_off_archetype'].shape}  (expect (4000, 4))")
print(f"b_def_archetype shape : {samples['b_def_archetype'].shape}  (expect (4000, 4))")
print()
print("Cell 6 complete.")

/var/folders/nd/7h3lcr8d2cjbxfmfqghczqz40000gn/T/ipykernel_41450/974205171.py:59: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(


Starting NUTS sampler (model_08 refit — 4 chains, 2000 warmup, 1000 samples)...
  N_teams       : 131
  N_games       : 3,214
  N_CONFERENCES : 10
  Prior changes : r_negbinom Gamma(4.0,0.5), sigma_attack/defense HalfNormal(0.25)
  Warmup change : 1000 -> 2000  [fix for sigma_attack R-hat=1.0108]



sample: 100%|████████████████████████████████████████████████████████████████| 3000/3000 [03:38<00:00, 13.73it/s, 127 steps of size 3.04e-02. acc. prob=0.93]



Fit complete. Wall-clock time : 914.8s
Divergences   : 0 / 4000
Energy array  : shape=(4000,)  (chains x samples for BFMI)

r_negbinom shape : (4000, 10)  (expect (4000, 10))
r_negbinom posterior means by conference:
  ACC                    mean=23.7801  std=3.3724
  American Athletic      mean=18.2964  std=2.6779
  Big 12                 mean=16.2958  std=2.1039
  Big Ten                mean=13.2437  std=1.7228
  Conference USA         mean=13.8537  std=2.0167
  Mid-American           mean=16.7681  std=2.5467
  Mountain West          mean=18.5040  std=2.8499
  Pac-12                 mean=21.3075  std=3.3850
  SEC                    mean=23.5128  std=3.4328
  Sun Belt               mean=17.3794  std=2.3271

mu_league     mean=3.1909
hfa_league    mean=0.0292
sp_weight     mean=0.0634
sigma_attack  mean=0.0239
sigma_defense mean=0.0659

b_off_archetype shape : (4000, 4)  (expect (4000, 4))
b_def_archetype shape : (4000, 4)  (expect (4000, 4))

Cell 6 complete.


In [7]:
# =============================================================================
# CELL 7 — Convergence diagnostics
#
# Required thresholds (all must pass before posterior checks proceed):
#   R-hat < 1.01        for all parameters
#   ESS_bulk >= 400     for all parameters
#   ESS_tail >= 400     for all parameters
#   0 divergences       (asserted in Cell 6)
#   BFMI > 0.3          for all 4 chains
#
# BFMI: energy shape from sequential CPU run is (4000,) — reshape to
# (4, 1000) to recover per-chain energy, then compute per-chain BFMI.
# BFMI = Var(energy_transition) / Var(energy) per chain.
# =============================================================================

import arviz as az
import numpy as np

# --- Convert to InferenceData ---
idata = az.from_numpyro(mcmc)

# --- Scalar and low-dim parameter list (identical to model_06/model_07) ---
scalar_vars = [
    "mu_league", "hfa_league",
    "r_negbinom",
    "sigma_conference", "sigma_attack", "sigma_defense", "sigma_hfa_team",
    "sp_weight", "rec_weight_sunbelt",
    "b_close_game_epa", "b_close_game_def_epa",
    "b_pregame_elo", "b_elo_sp_divergence", "b_last3_win_pct",
    "b_away_travel_distance", "b_away_tz_delta", "b_wind_chill",
    "b_rush_rate_std_downs", "b_rush_rate_pass_downs",
    "b_off_sack_rate_allowed", "b_def_sack_rate",
    "b_off_archetype", "b_def_archetype",
    "b_last3_off_epa_avg", "b_last3_def_epa_avg",
    "b_last3_pts_scored_avg", "b_last3_pts_allowed_avg",
    "b_days_since_last_game", "b_close_play_count_delta",
    "b_away_elevation_delta",
]

summary = az.summary(idata, var_names=scalar_vars, round_to=4)
print("=== Convergence summary (scalar and low-dim parameters) ===")
print(summary[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'ess_bulk', 'ess_tail', 'r_hat']].to_string())
print()

# --- R-hat threshold ---
r_hat_vals        = summary['r_hat']
ess_bulk_vals     = summary['ess_bulk']
ess_tail_vals     = summary['ess_tail']

r_hat_failures    = r_hat_vals[r_hat_vals >= 1.01]
ess_bulk_failures = ess_bulk_vals[ess_bulk_vals < 400]
ess_tail_failures = ess_tail_vals[ess_tail_vals < 400]

print("=== Threshold check: R-hat < 1.01 ===")
if len(r_hat_failures) == 0:
    print("  PASS — all parameters R-hat < 1.01")
else:
    print(f"  FAIL — {len(r_hat_failures)} parameter(s) at or above threshold:")
    print(r_hat_failures.to_string())
print()

print("=== Threshold check: ESS_bulk >= 400 ===")
if len(ess_bulk_failures) == 0:
    print("  PASS — all parameters ESS_bulk >= 400")
else:
    print(f"  FAIL — {len(ess_bulk_failures)} parameter(s) below threshold:")
    print(ess_bulk_failures.to_string())
print()

print("=== Threshold check: ESS_tail >= 400 ===")
if len(ess_tail_failures) == 0:
    print("  PASS — all parameters ESS_tail >= 400")
else:
    print(f"  FAIL — {len(ess_tail_failures)} parameter(s) below threshold:")
    print(ess_tail_failures.to_string())
print()

# --- Team-level array diagnostics ---
print("=== Team-level parameter diagnostics (max R-hat across 131 teams) ===")
for var in ["alpha_team_raw", "delta_team_raw", "hfa_team_raw"]:
    team_summary  = az.summary(idata, var_names=[var], round_to=4)
    max_rhat      = team_summary['r_hat'].max()
    min_ess_bulk  = team_summary['ess_bulk'].min()
    min_ess_tail  = team_summary['ess_tail'].min()
    rhat_status   = "PASS" if max_rhat < 1.01 else "FAIL"
    print(f"  {var:<20s}  max_rhat={max_rhat:.4f} [{rhat_status}]  "
          f"min_ess_bulk={min_ess_bulk:.0f}  min_ess_tail={min_ess_tail:.0f}")
print()

# --- BFMI — reshape (4000,) -> (4, 1000), compute per chain ---
# BFMI = Var(delta_energy) / Var(energy) where delta_energy[i] = energy[i] - energy[i-1]
print("=== BFMI check (threshold: > 0.3 per chain) ===")
energy_all = np.array(energy)  # (4000,) — sequential chains
energy_chains = energy_all.reshape(4, 1000)  # (4 chains, 1000 samples)

bfmi_pass = True
for chain_idx in range(4):
    e = energy_chains[chain_idx]
    delta_e = np.diff(e)
    bfmi_val = float(np.var(delta_e) / np.var(e))
    status = "PASS" if bfmi_val > 0.3 else "FAIL"
    if status == "FAIL":
        bfmi_pass = False
    print(f"  Chain {chain_idx}  BFMI={bfmi_val:.4f}  [{status}]")

print()
if bfmi_pass:
    print("  PASS — all chains BFMI > 0.3")
else:
    print("  FAIL — one or more chains BFMI <= 0.3. Stop and investigate.")
print()

# --- Divergences (confirmed 0 in Cell 6, restate for record) ---
print(f"=== Divergences : {n_divergences} / 4000  [{'PASS' if n_divergences == 0 else 'FAIL'}] ===")
print()

# --- Watch parameter detail ---
print("=== Watch parameter detail ===")
watch_vars = ["sigma_attack", "sigma_defense", "hfa_league", "mu_league"]
for var in watch_vars:
    if var in summary.index:
        row = summary.loc[var]
        print(f"  {var:<20s}  mean={row['mean']:.4f}  "
              f"ess_bulk={row['ess_bulk']:.0f}  ess_tail={row['ess_tail']:.0f}  "
              f"r_hat={row['r_hat']:.4f}")
print()
print("Cell 7 complete.")

=== Convergence summary (scalar and low-dim parameters) ===
                             mean      sd   hdi_3%  hdi_97%   ess_bulk   ess_tail   r_hat
mu_league                  3.1909  0.0958   3.0060   3.3639  1367.5794  1795.0571  1.0049
hfa_league                 0.0292  0.0116   0.0083   0.0508  6438.8779  3463.4184  1.0013
r_negbinom[0]             23.7800  3.3729  17.5299  29.9488  6192.7394  3078.8549  0.9996
r_negbinom[1]             18.2964  2.6783  13.3682  23.1325  6303.2885  2742.7642  1.0010
r_negbinom[2]             16.2958  2.1041  12.5835  20.2927  6762.5025  2978.8773  0.9999
r_negbinom[3]             13.2437  1.7230  10.1294  16.5153  6900.9384  2918.8077  1.0005
r_negbinom[4]             13.8537  2.0170  10.2292  17.6686  6788.0509  2712.5398  1.0003
r_negbinom[5]             16.7681  2.5471  12.1566  21.5992  7156.5986  3072.3037  1.0014
r_negbinom[6]             18.5039  2.8502  13.5521  24.0145  5846.9892  2997.5950  1.0000
r_negbinom[7]             21.3075  3.385

In [8]:
# =============================================================================
# CELL 8 — Trace plots
#
# Watch parameters: sigma_attack, hfa_league, mu_league, r_negbinom[0,8]
# Mirrors model_07 Cell 3 pattern. Saved to artifacts/model_08_trace_plots.png
#
# sigma_attack is the primary watch parameter: near-boundary HalfNormal,
# ESS_bulk ~538 across both runs. Trace plot documents chain behavior
# for the session record even though R-hat < 1.01.
# =============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import arviz as az
import os

TRACE_PATH = os.path.join(ARTIFACTS_DIR, "model_08_trace_plots.png")

trace_vars = ["sigma_attack", "hfa_league", "mu_league", "r_negbinom"]

ax = az.plot_trace(
    idata,
    var_names=trace_vars,
    compact=True,
    figsize=(14, 10),
)

plt.suptitle(
    "model_08 trace plots — sigma_attack, hfa_league, mu_league, r_negbinom\n"
    "Priors: r~Gamma(4,0.5), sigma_attack/defense~HalfNormal(0.25)",
    fontsize=10, y=1.01
)
plt.tight_layout()
plt.savefig(TRACE_PATH, dpi=120, bbox_inches='tight')
plt.close()

assert os.path.exists(TRACE_PATH), f"Trace plot not saved: {TRACE_PATH}"
print(f"Trace plot saved : {TRACE_PATH}")
print(f"  Size           : {os.path.getsize(TRACE_PATH) / 1e3:.1f} KB")
print()

# --- Summarize chain spread for watch parameters ---
print("=== Chain spread check (posterior mean per chain) ===")
watch_detail = {
    "sigma_attack" : samples['sigma_attack'],   # (4000,)
    "hfa_league"   : samples['hfa_league'],
    "mu_league"    : samples['mu_league'],
}
# samples are concatenated (4 chains x 1000) = 4000
for name, arr in watch_detail.items():
    chains = arr.reshape(4, 1000)
    chain_means = [float(chains[c].mean()) for c in range(4)]
    spread = max(chain_means) - min(chain_means)
    print(f"  {name:<20s}  chain means={[f'{m:.4f}' for m in chain_means]}  "
          f"spread={spread:.4f}")

# r_negbinom[0]=ACC, r_negbinom[8]=SEC
r_chains = samples['r_negbinom'].reshape(4, 1000, N_CONFERENCES)
for conf_idx_watch, conf_name in [(0, 'ACC'), (8, 'SEC')]:
    chain_means = [float(r_chains[c, :, conf_idx_watch].mean()) for c in range(4)]
    spread = max(chain_means) - min(chain_means)
    print(f"  r_negbinom[{conf_idx_watch}] ({conf_name:<4s})  "
          f"chain means={[f'{m:.4f}' for m in chain_means]}  spread={spread:.4f}")

print()
print("Cell 8 complete.")

Trace plot saved : /Users/kevinjohnson/cfb-analytics/artifacts/model_08_trace_plots.png
  Size           : 923.6 KB

=== Chain spread check (posterior mean per chain) ===
  sigma_attack          chain means=['0.0237', '0.0252', '0.0236', '0.0232']  spread=0.0020
  hfa_league            chain means=['0.0293', '0.0290', '0.0295', '0.0292']  spread=0.0005
  mu_league             chain means=['3.1815', '3.1923', '3.1945', '3.1954']  spread=0.0140
  r_negbinom[0] (ACC )  chain means=['23.7822', '23.7290', '23.8553', '23.7538']  spread=0.1263
  r_negbinom[8] (SEC )  chain means=['23.4439', '23.5726', '23.5561', '23.4788']  spread=0.1287

Cell 8 complete.


In [9]:
# =============================================================================
# CELL 9 — Posterior predictive VMR check
#
# Required: observed VMR inside 90% CI per conference (all 10 must pass).
# model_07 Finding 1: all 10 conferences flagged. Observed VMR 4.8-7.2,
# posterior predictive CI 2.1-4.0. Fix: r_negbinom Gamma(4.0, 0.5).
#
# Method: 200 posterior draws. Per draw, compute realized scores from
# NegBin2(mu, r) via Gamma-Poisson. Compute VMR per conference per draw.
# Check observed VMR against 5th-95th percentile of draw distribution.
#
# Observed scores and conference labels come from df loaded in Cell 3.
# Posterior arrays extracted from samples dict (Cell 6).
# =============================================================================

import numpy as np

# --- Extract posterior arrays ---
mu_league_s  = np.array(samples['mu_league'])         # (4000,)
hfa_league_s = np.array(samples['hfa_league'])         # (4000,)
mu_conf_s    = np.array(samples['mu_conference'])      # (4000, 10)
alpha_s      = np.array(samples['alpha_team'])         # (4000, 131)
delta_s      = np.array(samples['delta_team'])         # (4000, 131)
hfa_s        = np.array(samples['hfa_team'])           # (4000, 131)
r_s          = np.array(samples['r_negbinom'])         # (4000, 10)

# --- Build observation-level index arrays (numpy) ---
team_idx_arr  = np.array(team_idx,  dtype=np.int32)
opp_idx_arr   = np.array(opp_idx,   dtype=np.int32)
conf_idx_arr  = np.array(conf_idx,  dtype=np.int32)
is_home_arr   = np.array(df['is_home'].values,       dtype=np.float32)
points_arr    = np.array(df['points_scored'].values,  dtype=np.float32)
conf_labels   = np.array(df['conference'].values)

# --- Observed VMR per conference ---
obs_vmr = {}
for c_name in CONFERENCES:
    mask = conf_labels == c_name
    y_c  = points_arr[mask]
    mean_c = y_c.mean()
    if mean_c > 1e-6:
        obs_vmr[c_name] = float(y_c.var() / mean_c)

print(f"r_negbinom shape : {r_s.shape}")
print(f"df rows          : {len(df)}")
print(f"obs_vmr conferences: {list(obs_vmr.keys())}")
print()

# --- 200 posterior draws ---
rng      = np.random.default_rng(42)
draw_idx = rng.choice(4000, size=200, replace=False)

conf_vmr_draws = {c: [] for c in CONFERENCES}

for d in draw_idx:
    log_mu = (
        mu_league_s[d]
        + mu_conf_s[d, conf_idx_arr]
        + alpha_s[d, team_idx_arr]
        - delta_s[d, opp_idx_arr]
        + hfa_league_s[d] * is_home_arr
        + hfa_s[d, team_idx_arr] * is_home_arr
    )
    mu_pred = np.exp(log_mu).clip(1e-6)

    # NegBin2(mu, r) via Gamma-Poisson:
    #   lambda ~ Gamma(shape=r, scale=mu/r)
    #   y      ~ Poisson(lambda)
    r_per_obs  = r_s[d, conf_idx_arr]                          # (N_obs,)
    gamma_draw = rng.gamma(shape=r_per_obs, scale=mu_pred / r_per_obs)
    y_draw     = rng.poisson(lam=gamma_draw).astype(float)

    for c_name in CONFERENCES:
        mask = conf_labels == c_name
        if mask.sum() < 5:
            continue
        y_c    = y_draw[mask]
        mean_c = y_c.mean()
        if mean_c < 1e-6:
            continue
        conf_vmr_draws[c_name].append(y_c.var() / mean_c)

# --- 90% CI vs observed ---
print("=== Posterior predictive VMR — 90% CI vs observed ===")
print(f"  {'Conference':<22s}  {'Obs VMR':>8s}  {'PP 5%':>8s}  {'PP 95%':>8s}  Status")
print("  " + "-" * 62)
all_pass = True
for c_name in CONFERENCES:
    draws_c = np.array(conf_vmr_draws[c_name])
    if len(draws_c) == 0:
        continue
    lo  = np.percentile(draws_c, 5)
    hi  = np.percentile(draws_c, 95)
    obs = obs_vmr.get(c_name, np.nan)
    if np.isnan(obs):
        status = "NO OBS"
    elif lo <= obs <= hi:
        status = "PASS"
    else:
        status = "FLAG"
        all_pass = False
    print(f"  {c_name:<22s}  {obs:>8.4f}  {lo:>8.4f}  {hi:>8.4f}  {status}")

print()
if all_pass:
    print("  PASS — all conferences inside 90% CI")
else:
    print("  FLAG — one or more conferences outside 90% CI")
print()

# --- Implied VMR at posterior means (diagnostic) ---
r_mean       = r_s.mean(axis=0)   # (10,)
log_mu_mean  = (
    mu_league_s.mean()
    + mu_conf_s.mean(axis=0)[conf_idx_arr]
    + alpha_s.mean(axis=0)[team_idx_arr]
    - delta_s.mean(axis=0)[opp_idx_arr]
    + hfa_league_s.mean() * is_home_arr
    + hfa_s.mean(axis=0)[team_idx_arr] * is_home_arr
)
mu_mean_pred = np.exp(log_mu_mean).clip(1e-6)

print("=== Implied VMR from NegBin2 at posterior means (VMR = 1 + mu/r) ===")
print(f"  {'Conference':<22s}  {'mu_mean':>8s}  {'r_mean':>8s}  "
      f"{'Implied VMR':>12s}  {'Obs VMR':>8s}  {'Gap':>8s}")
print("  " + "-" * 75)
for c_name in CONFERENCES:
    mask        = conf_labels == c_name
    mu_c        = mu_mean_pred[mask].mean()
    r_c         = r_mean[conf_to_idx[c_name]]
    vmr_implied = 1 + mu_c / r_c
    obs         = obs_vmr.get(c_name, np.nan)
    gap         = obs - vmr_implied
    print(f"  {c_name:<22s}  {mu_c:>8.2f}  {r_c:>8.2f}  "
          f"{vmr_implied:>12.4f}  {obs:>8.4f}  {gap:>8.4f}")
print()
print("Cell 9 complete.")

r_negbinom shape : (4000, 10)
df rows          : 3214
obs_vmr conferences: ['ACC', 'American Athletic', 'Big 12', 'Big Ten', 'Conference USA', 'Mid-American', 'Mountain West', 'Pac-12', 'SEC', 'Sun Belt']

=== Posterior predictive VMR — 90% CI vs observed ===
  Conference               Obs VMR     PP 5%    PP 95%  Status
  --------------------------------------------------------------
  ACC                       4.8275    1.7972    2.6330  FLAG
  American Athletic         7.0632    2.1746    3.2039  FLAG
  Big 12                    5.6139    2.0749    3.0560  FLAG
  Big Ten                   7.1412    2.3071    3.6058  FLAG
  Conference USA            6.0558    2.3066    3.6794  FLAG
  Mid-American              5.5069    2.2159    3.5401  FLAG
  Mountain West             5.7427    2.0296    3.2365  FLAG
  Pac-12                    6.3096    1.7951    2.8353  FLAG
  SEC                       6.0788    1.6925    2.5168  FLAG
  Sun Belt                  5.5668    2.1224    3.1733  FLAG

 

In [10]:
# =============================================================================
# CELL 10 (REWRITE) — Diagnostic model comparison
#
# Same four models A/B/C/D. Two changes from previous run:
#   num_warmup  : 500  -> 1000  (previous warmup insufficient for geometry)
#   num_samples : 500  -> 1000  (more samples for stable variance/mean checks)
#
# R-hat will still be NaN (single chain) — that is expected and noted.
# ESS_bulk and variance checks should now be interpretable.
# All model definitions, init values, and check logic unchanged.
# =============================================================================

import numpy as np
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value
from jax import random
import arviz as az
import time

# ---------------------------------------------------------------------------
# Shared: log(points + 1) target for Log-Normal models
# ---------------------------------------------------------------------------
log_points = jnp.array(
    np.log1p(np.array(df['points_scored'].values, dtype=np.float32)),
    dtype=jnp.float32
)

conf_labels   = np.array(df['conference'].values)
log_points_np = np.array(log_points)
points_np     = np.array(df['points_scored'].values, dtype=np.float32)
obs_log_var   = {}
obs_vmr       = {}
for c_name in CONFERENCES:
    mask = conf_labels == c_name
    y_log = log_points_np[mask]
    y_raw = points_np[mask]
    obs_log_var[c_name] = float(y_log.var())
    mean_c = y_raw.mean()
    if mean_c > 1e-6:
        obs_vmr[c_name] = float(y_raw.var() / mean_c)

team_idx_arr = np.array(team_idx,  dtype=np.int32)
opp_idx_arr  = np.array(opp_idx,   dtype=np.int32)
conf_idx_arr = np.array(conf_idx,  dtype=np.int32)
is_home_arr  = np.array(df['is_home'].values, dtype=np.float32)

# ---------------------------------------------------------------------------
# Model definitions — identical to previous Cell 10
# (model_cfb_b, model_cfb_c, model_cfb_d already defined above,
#  but redefined here in full so this cell is self-contained)
# ---------------------------------------------------------------------------

def model_cfb_b(data: GameData, N_teams: int):
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))
    log_r_mu    = numpyro.sample("log_r_mu",    dist.Normal(jnp.log(8.0), 0.5))
    log_r_sigma = numpyro.sample("log_r_sigma", dist.HalfNormal(0.3))
    log_r_raw   = numpyro.sample(
        "log_r_raw", dist.Normal(0.0, 1.0), sample_shape=(N_CONFERENCES,)
    )
    r_negbinom = numpyro.deterministic(
        "r_negbinom", jnp.exp(log_r_mu + log_r_sigma * log_r_raw).clip(1e-6)
    )
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(0.1))
    mu_conference    = numpyro.sample(
        "mu_conference", dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )
    sigma_attack   = numpyro.sample("sigma_attack",   dist.HalfNormal(0.25))
    alpha_team_raw = numpyro.sample(
        "alpha_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    alpha_team = numpyro.deterministic("alpha_team", alpha_team_raw * sigma_attack)
    sigma_defense  = numpyro.sample("sigma_defense",  dist.HalfNormal(0.25))
    delta_team_raw = numpyro.sample(
        "delta_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    delta_team = numpyro.deterministic("delta_team", delta_team_raw * sigma_defense)
    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team_raw   = numpyro.sample(
        "hfa_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    hfa_team = numpyro.deterministic("hfa_team", hfa_team_raw * sigma_hfa_team)
    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 0.15))
    rec_weight_other   = numpyro.sample(
        "rec_weight_other", dist.Normal(0.0, 0.5), sample_shape=(N_CONFERENCES - 1,)
    )
    rec_weight_sunbelt = numpyro.sample(
        "rec_weight_sunbelt", dist.TruncatedNormal(0.0, 0.5, high=0.0)
    )
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])
    b_close_game_epa       = numpyro.sample("b_close_game_epa",       dist.Normal(0.0, 0.10))
    b_close_game_def_epa   = numpyro.sample("b_close_game_def_epa",   dist.Normal(0.0, 0.10))
    b_pregame_elo          = numpyro.sample("b_pregame_elo",          dist.Normal(0.0, 0.15))
    b_elo_sp_divergence    = numpyro.sample("b_elo_sp_divergence",    dist.Normal(0.0, 0.15))
    b_last3_win_pct        = numpyro.sample("b_last3_win_pct",        dist.Normal(0.0, 0.15))
    b_away_travel_distance = numpyro.sample("b_away_travel_distance", dist.Normal(0.0, 0.15))
    b_away_tz_delta        = numpyro.sample("b_away_tz_delta",        dist.Normal(0.0, 0.15))
    b_wind_chill           = numpyro.sample("b_wind_chill",           dist.Normal(0.0, 0.15))
    b_rush_rate_std_downs  = numpyro.sample("b_rush_rate_std_downs",  dist.Normal(0.0, 0.15))
    b_rush_rate_pass_downs = numpyro.sample("b_rush_rate_pass_downs", dist.Normal(0.0, 0.15))
    b_off_sack_rate_allowed= numpyro.sample("b_off_sack_rate_allowed",dist.Normal(0.0, 0.15))
    b_def_sack_rate        = numpyro.sample("b_def_sack_rate",        dist.Normal(0.0, 0.15))
    b_off_archetype = numpyro.sample(
        "b_off_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_def_archetype = numpyro.sample(
        "b_def_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_last3_off_epa_avg      = numpyro.sample("b_last3_off_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_def_epa_avg      = numpyro.sample("b_last3_def_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_pts_scored_avg   = numpyro.sample("b_last3_pts_scored_avg",   dist.Normal(0.0, 0.15))
    b_last3_pts_allowed_avg  = numpyro.sample("b_last3_pts_allowed_avg",  dist.Normal(0.0, 0.15))
    b_days_since_last_game   = numpyro.sample("b_days_since_last_game",   dist.Normal(0.0, 0.15))
    b_close_play_count_delta = numpyro.sample("b_close_play_count_delta", dist.Normal(0.0, 0.15))
    b_away_elevation_delta   = numpyro.sample("b_away_elevation_delta",   dist.Normal(0.0, 0.15))
    log_mu = (
        mu_league
        + mu_conference[data.conf_idx]
        + alpha_team[data.team_idx]
        - delta_team[data.opp_idx]
        + (hfa_league + hfa_team[data.team_idx]) * data.is_home
        + sp_weight              * data.sp_rating
        + rec_weight[data.conf_idx] * data.recruiting_3yr_avg
        + b_close_game_epa       * data.close_game_epa
        + b_close_game_def_epa   * data.close_game_def_epa
        + b_pregame_elo          * data.pregame_elo
        + b_elo_sp_divergence    * data.elo_sp_divergence
        + b_last3_win_pct        * data.last3_win_pct
        + b_away_travel_distance * data.away_travel_distance
        + b_away_tz_delta        * data.away_tz_delta
        + b_wind_chill           * data.wind_chill
        + b_rush_rate_std_downs  * data.rush_rate_std_downs
        + b_rush_rate_pass_downs * data.rush_rate_pass_downs
        + b_off_sack_rate_allowed * data.off_sack_rate_allowed
        + b_def_sack_rate         * data.def_sack_rate
        + b_off_archetype[data.off_archetype_idx]
        + b_def_archetype[data.def_archetype_idx]
        + b_last3_off_epa_avg     * data.mask_last3_off_epa    * data.last3_off_epa_avg
        + b_last3_def_epa_avg     * data.mask_last3_def_epa    * data.last3_def_epa_avg
        + b_last3_pts_scored_avg  * data.mask_last3_pts_scored * data.last3_pts_scored_avg
        + b_last3_pts_allowed_avg * data.mask_last3_pts_allowed * data.last3_pts_allowed_avg
        + b_days_since_last_game  * data.mask_days_since        * data.days_since_last_game
        + b_close_play_count_delta * data.mask_close_play_count * data.close_play_count_delta
        + b_away_elevation_delta  * data.mask_elevation         * data.away_elevation_delta
    )
    mu = jnp.exp(log_mu).clip(1e-6)
    r  = r_negbinom[data.conf_idx].clip(1e-6)
    numpyro.sample("obs", dist.NegativeBinomial2(mu, r, validate_args=False),
                   obs=data.points)


def _log_mu_block(data, mu_league, hfa_league, mu_conference, alpha_team,
                  delta_team, hfa_team, sp_weight, rec_weight,
                  b_close_game_epa, b_close_game_def_epa, b_pregame_elo,
                  b_elo_sp_divergence, b_last3_win_pct, b_away_travel_distance,
                  b_away_tz_delta, b_wind_chill, b_rush_rate_std_downs,
                  b_rush_rate_pass_downs, b_off_sack_rate_allowed, b_def_sack_rate,
                  b_off_archetype, b_def_archetype, b_last3_off_epa_avg,
                  b_last3_def_epa_avg, b_last3_pts_scored_avg, b_last3_pts_allowed_avg,
                  b_days_since_last_game, b_close_play_count_delta, b_away_elevation_delta):
    return (
        mu_league
        + mu_conference[data.conf_idx]
        + alpha_team[data.team_idx]
        - delta_team[data.opp_idx]
        + (hfa_league + hfa_team[data.team_idx]) * data.is_home
        + sp_weight              * data.sp_rating
        + rec_weight[data.conf_idx] * data.recruiting_3yr_avg
        + b_close_game_epa       * data.close_game_epa
        + b_close_game_def_epa   * data.close_game_def_epa
        + b_pregame_elo          * data.pregame_elo
        + b_elo_sp_divergence    * data.elo_sp_divergence
        + b_last3_win_pct        * data.last3_win_pct
        + b_away_travel_distance * data.away_travel_distance
        + b_away_tz_delta        * data.away_tz_delta
        + b_wind_chill           * data.wind_chill
        + b_rush_rate_std_downs  * data.rush_rate_std_downs
        + b_rush_rate_pass_downs * data.rush_rate_pass_downs
        + b_off_sack_rate_allowed * data.off_sack_rate_allowed
        + b_def_sack_rate         * data.def_sack_rate
        + b_off_archetype[data.off_archetype_idx]
        + b_def_archetype[data.def_archetype_idx]
        + b_last3_off_epa_avg     * data.mask_last3_off_epa    * data.last3_off_epa_avg
        + b_last3_def_epa_avg     * data.mask_last3_def_epa    * data.last3_def_epa_avg
        + b_last3_pts_scored_avg  * data.mask_last3_pts_scored * data.last3_pts_scored_avg
        + b_last3_pts_allowed_avg * data.mask_last3_pts_allowed * data.last3_pts_allowed_avg
        + b_days_since_last_game  * data.mask_days_since        * data.days_since_last_game
        + b_close_play_count_delta * data.mask_close_play_count * data.close_play_count_delta
        + b_away_elevation_delta  * data.mask_elevation         * data.away_elevation_delta
    )


def _sample_common_params(N_teams):
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(0.1))
    mu_conference    = numpyro.sample(
        "mu_conference", dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )
    sigma_attack   = numpyro.sample("sigma_attack",   dist.HalfNormal(0.25))
    alpha_team_raw = numpyro.sample(
        "alpha_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    alpha_team = numpyro.deterministic("alpha_team", alpha_team_raw * sigma_attack)
    sigma_defense  = numpyro.sample("sigma_defense",  dist.HalfNormal(0.25))
    delta_team_raw = numpyro.sample(
        "delta_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    delta_team = numpyro.deterministic("delta_team", delta_team_raw * sigma_defense)
    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team_raw   = numpyro.sample(
        "hfa_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    hfa_team = numpyro.deterministic("hfa_team", hfa_team_raw * sigma_hfa_team)
    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 0.15))
    rec_weight_other   = numpyro.sample(
        "rec_weight_other", dist.Normal(0.0, 0.5), sample_shape=(N_CONFERENCES - 1,)
    )
    rec_weight_sunbelt = numpyro.sample(
        "rec_weight_sunbelt", dist.TruncatedNormal(0.0, 0.5, high=0.0)
    )
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])
    b_close_game_epa       = numpyro.sample("b_close_game_epa",       dist.Normal(0.0, 0.10))
    b_close_game_def_epa   = numpyro.sample("b_close_game_def_epa",   dist.Normal(0.0, 0.10))
    b_pregame_elo          = numpyro.sample("b_pregame_elo",          dist.Normal(0.0, 0.15))
    b_elo_sp_divergence    = numpyro.sample("b_elo_sp_divergence",    dist.Normal(0.0, 0.15))
    b_last3_win_pct        = numpyro.sample("b_last3_win_pct",        dist.Normal(0.0, 0.15))
    b_away_travel_distance = numpyro.sample("b_away_travel_distance", dist.Normal(0.0, 0.15))
    b_away_tz_delta        = numpyro.sample("b_away_tz_delta",        dist.Normal(0.0, 0.15))
    b_wind_chill           = numpyro.sample("b_wind_chill",           dist.Normal(0.0, 0.15))
    b_rush_rate_std_downs  = numpyro.sample("b_rush_rate_std_downs",  dist.Normal(0.0, 0.15))
    b_rush_rate_pass_downs = numpyro.sample("b_rush_rate_pass_downs", dist.Normal(0.0, 0.15))
    b_off_sack_rate_allowed= numpyro.sample("b_off_sack_rate_allowed",dist.Normal(0.0, 0.15))
    b_def_sack_rate        = numpyro.sample("b_def_sack_rate",        dist.Normal(0.0, 0.15))
    b_off_archetype = numpyro.sample(
        "b_off_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_def_archetype = numpyro.sample(
        "b_def_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_last3_off_epa_avg      = numpyro.sample("b_last3_off_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_def_epa_avg      = numpyro.sample("b_last3_def_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_pts_scored_avg   = numpyro.sample("b_last3_pts_scored_avg",   dist.Normal(0.0, 0.15))
    b_last3_pts_allowed_avg  = numpyro.sample("b_last3_pts_allowed_avg",  dist.Normal(0.0, 0.15))
    b_days_since_last_game   = numpyro.sample("b_days_since_last_game",   dist.Normal(0.0, 0.15))
    b_close_play_count_delta = numpyro.sample("b_close_play_count_delta", dist.Normal(0.0, 0.15))
    b_away_elevation_delta   = numpyro.sample("b_away_elevation_delta",   dist.Normal(0.0, 0.15))
    return (mu_league, hfa_league, mu_conference, alpha_team, delta_team,
            hfa_team, sp_weight, rec_weight,
            b_close_game_epa, b_close_game_def_epa, b_pregame_elo,
            b_elo_sp_divergence, b_last3_win_pct, b_away_travel_distance,
            b_away_tz_delta, b_wind_chill, b_rush_rate_std_downs,
            b_rush_rate_pass_downs, b_off_sack_rate_allowed, b_def_sack_rate,
            b_off_archetype, b_def_archetype, b_last3_off_epa_avg,
            b_last3_def_epa_avg, b_last3_pts_scored_avg, b_last3_pts_allowed_avg,
            b_days_since_last_game, b_close_play_count_delta, b_away_elevation_delta)


def model_cfb_c(data: GameData, N_teams: int, log_points: jnp.ndarray):
    sigma_obs = numpyro.sample(
        "sigma_obs", dist.HalfNormal(0.3), sample_shape=(N_CONFERENCES,)
    )
    params = _sample_common_params(N_teams)
    log_mu = _log_mu_block(data, *params)
    numpyro.sample("obs", dist.Normal(log_mu, sigma_obs[data.conf_idx]), obs=log_points)


def model_cfb_d(data: GameData, N_teams: int, log_points: jnp.ndarray):
    sigma_mu  = numpyro.sample("sigma_mu",  dist.HalfNormal(0.3))
    sigma_raw = numpyro.sample(
        "sigma_raw", dist.HalfNormal(1.0), sample_shape=(N_CONFERENCES,)
    )
    sigma_obs = numpyro.deterministic("sigma_obs", sigma_mu * sigma_raw)
    params = _sample_common_params(N_teams)
    log_mu = _log_mu_block(data, *params)
    numpyro.sample("obs", dist.Normal(log_mu, sigma_obs[data.conf_idx]), obs=log_points)


# ---------------------------------------------------------------------------
# Init values
# ---------------------------------------------------------------------------
base_init = {
    "mu_league"          : jnp.array(3.3),
    "hfa_league"         : jnp.array(0.1),
    "sigma_conference"   : jnp.array(0.05),
    "mu_conference"      : jnp.zeros(N_CONFERENCES),
    "sigma_attack"       : jnp.array(0.05),
    "alpha_team_raw"     : jnp.zeros(N_teams),
    "sigma_defense"      : jnp.array(0.05),
    "delta_team_raw"     : jnp.zeros(N_teams),
    "sigma_hfa_team"     : jnp.array(0.05),
    "hfa_team_raw"       : jnp.zeros(N_teams),
    "sp_weight"          : jnp.array(0.0),
    "rec_weight_other"   : jnp.zeros(N_CONFERENCES - 1),
    "rec_weight_sunbelt" : jnp.array(-0.1),
    "b_close_game_epa"        : jnp.array(0.0),
    "b_close_game_def_epa"    : jnp.array(0.0),
    "b_pregame_elo"           : jnp.array(0.0),
    "b_elo_sp_divergence"     : jnp.array(0.0),
    "b_last3_win_pct"         : jnp.array(0.0),
    "b_away_travel_distance"  : jnp.array(0.0),
    "b_away_tz_delta"         : jnp.array(0.0),
    "b_wind_chill"            : jnp.array(0.0),
    "b_rush_rate_std_downs"   : jnp.array(0.0),
    "b_rush_rate_pass_downs"  : jnp.array(0.0),
    "b_off_sack_rate_allowed" : jnp.array(0.0),
    "b_def_sack_rate"         : jnp.array(0.0),
    "b_off_archetype"         : jnp.zeros(4),
    "b_def_archetype"         : jnp.zeros(4),
    "b_last3_off_epa_avg"     : jnp.array(0.0),
    "b_last3_def_epa_avg"     : jnp.array(0.0),
    "b_last3_pts_scored_avg"  : jnp.array(0.0),
    "b_last3_pts_allowed_avg" : jnp.array(0.0),
    "b_days_since_last_game"  : jnp.array(0.0),
    "b_close_play_count_delta": jnp.array(0.0),
    "b_away_elevation_delta"  : jnp.array(0.0),
}

init_a = {**base_init, "r_negbinom": jnp.ones(N_CONFERENCES) * 8.0}
init_b = {**base_init,
          "log_r_mu"   : jnp.array(float(jnp.log(8.0))),
          "log_r_sigma": jnp.array(0.1),
          "log_r_raw"  : jnp.zeros(N_CONFERENCES)}
init_c = {**base_init, "sigma_obs": jnp.ones(N_CONFERENCES) * 0.3}
init_d = {**base_init,
          "sigma_mu" : jnp.array(0.3),
          "sigma_raw": jnp.ones(N_CONFERENCES) * 1.0}


# ---------------------------------------------------------------------------
# Fit function — 1 chain, 1000 warmup, 1000 samples
# ---------------------------------------------------------------------------
def run_diagnostic(label, model_fn, init_vals, model_kwargs):
    print(f"  Fitting Model {label}...", end=" ", flush=True)
    nuts  = NUTS(model_fn, target_accept_prob=0.9,
                 init_strategy=init_to_value(values=init_vals))
    mcmc_ = MCMC(nuts, num_warmup=1000, num_samples=1000,
                 num_chains=1, progress_bar=False)
    numpyro.enable_validation(False)
    try:
        t0 = time.time()
        mcmc_.run(random.PRNGKey(0), **model_kwargs)
        elapsed = time.time() - t0
    finally:
        numpyro.enable_validation(True)
    extra_ = mcmc_.get_extra_fields()
    n_div  = int(extra_['diverging'].sum()) if 'diverging' in extra_ else 0
    samp_  = mcmc_.get_samples()
    idata_ = az.from_numpyro(mcmc_)
    print(f"done ({elapsed:.0f}s)  divergences={n_div}")
    return samp_, idata_, n_div, elapsed


print("=" * 60)
print("Running diagnostic fits (1 chain, 1000 warmup, 1000 samples)")
print("=" * 60)

samp_a, idata_a, ndiv_a, t_a = run_diagnostic(
    "A", model_cfb,
    init_a,
    {"data": training_data, "N_teams": N_teams}
)
samp_b, idata_b, ndiv_b, t_b = run_diagnostic(
    "B", model_cfb_b,
    init_b,
    {"data": training_data, "N_teams": N_teams}
)
samp_c, idata_c, ndiv_c, t_c = run_diagnostic(
    "C", model_cfb_c,
    init_c,
    {"data": training_data, "N_teams": N_teams, "log_points": log_points}
)
samp_d, idata_d, ndiv_d, t_d = run_diagnostic(
    "D", model_cfb_d,
    init_d,
    {"data": training_data, "N_teams": N_teams, "log_points": log_points}
)
print()

# ---------------------------------------------------------------------------
# ESS_bulk — use az.ess directly, which works on single-chain output
# ---------------------------------------------------------------------------
def get_min_ess(idata_):
    scalar_vars = [
        "mu_league", "hfa_league", "sigma_conference",
        "sigma_attack", "sigma_defense", "sigma_hfa_team",
        "sp_weight", "b_close_game_epa",
    ]
    ess = az.ess(idata_, var_names=scalar_vars)
    vals = [float(ess[v].values.min()) for v in scalar_vars if v in ess]
    return min(vals)

ess_a = get_min_ess(idata_a)
ess_b = get_min_ess(idata_b)
ess_c = get_min_ess(idata_c)
ess_d = get_min_ess(idata_d)

# ---------------------------------------------------------------------------
# Variance check — VMR (A,B) and log-variance (C,D) — 100 draws
# ---------------------------------------------------------------------------
def vmr_check_negbin(samp_):
    mu_l  = np.array(samp_['mu_league'])
    hfa_l = np.array(samp_['hfa_league'])
    mu_c  = np.array(samp_['mu_conference'])
    alp   = np.array(samp_['alpha_team'])
    dlt   = np.array(samp_['delta_team'])
    hfa   = np.array(samp_['hfa_team'])
    r_s   = np.array(samp_['r_negbinom'])
    rng   = np.random.default_rng(0)
    draws = rng.choice(len(mu_l), size=100, replace=False)
    conf_draws = {c: [] for c in CONFERENCES}
    for d in draws:
        log_mu = (mu_l[d] + mu_c[d, conf_idx_arr] + alp[d, team_idx_arr]
                  - dlt[d, opp_idx_arr]
                  + hfa_l[d] * is_home_arr + hfa[d, team_idx_arr] * is_home_arr)
        mu_p = np.exp(log_mu).clip(1e-6)
        r_o  = r_s[d, conf_idx_arr]
        g    = rng.gamma(shape=r_o, scale=mu_p / r_o)
        y    = rng.poisson(lam=g).astype(float)
        for c in CONFERENCES:
            mask = conf_labels == c
            yc = y[mask]; mc = yc.mean()
            if mc > 1e-6:
                conf_draws[c].append(yc.var() / mc)
    n_pass = 0
    for c in CONFERENCES:
        dc = np.array(conf_draws[c])
        lo, hi = np.percentile(dc, 5), np.percentile(dc, 95)
        if lo <= obs_vmr[c] <= hi:
            n_pass += 1
    return n_pass


def logvar_check_lognormal(samp_):
    mu_l  = np.array(samp_['mu_league'])
    hfa_l = np.array(samp_['hfa_league'])
    mu_c  = np.array(samp_['mu_conference'])
    alp   = np.array(samp_['alpha_team'])
    dlt   = np.array(samp_['delta_team'])
    hfa   = np.array(samp_['hfa_team'])
    sig   = np.array(samp_['sigma_obs'])
    rng   = np.random.default_rng(0)
    draws = rng.choice(len(mu_l), size=100, replace=False)
    conf_draws = {c: [] for c in CONFERENCES}
    for d in draws:
        log_mu = (mu_l[d] + mu_c[d, conf_idx_arr] + alp[d, team_idx_arr]
                  - dlt[d, opp_idx_arr]
                  + hfa_l[d] * is_home_arr + hfa[d, team_idx_arr] * is_home_arr)
        sig_o  = sig[d, conf_idx_arr]
        y_log  = rng.normal(loc=log_mu, scale=sig_o)
        for c in CONFERENCES:
            mask = conf_labels == c
            conf_draws[c].append(y_log[mask].var())
    n_pass = 0
    for c in CONFERENCES:
        dc = np.array(conf_draws[c])
        lo, hi = np.percentile(dc, 5), np.percentile(dc, 95)
        if lo <= obs_log_var[c] <= hi:
            n_pass += 1
    return n_pass


print("Running variance checks...", flush=True)
vpass_a = vmr_check_negbin(samp_a)
vpass_b = vmr_check_negbin(samp_b)
vpass_c = logvar_check_lognormal(samp_c)
vpass_d = logvar_check_lognormal(samp_d)
print("done.")
print()

# ---------------------------------------------------------------------------
# Team-level and conference-level mean checks
# ---------------------------------------------------------------------------
def mean_checks(samp_, is_lognormal=False):
    mu_l  = np.array(samp_['mu_league']).mean()
    hfa_l = np.array(samp_['hfa_league']).mean()
    mu_c  = np.array(samp_['mu_conference']).mean(axis=0)
    alp   = np.array(samp_['alpha_team']).mean(axis=0)
    dlt   = np.array(samp_['delta_team']).mean(axis=0)
    hfa   = np.array(samp_['hfa_team']).mean(axis=0)
    log_mu = (mu_l + mu_c[conf_idx_arr] + alp[team_idx_arr]
              - dlt[opp_idx_arr]
              + hfa_l * is_home_arr + hfa[team_idx_arr] * is_home_arr)
    if is_lognormal:
        sig   = np.array(samp_['sigma_obs']).mean(axis=0)
        sig_o = sig[conf_idx_arr]
        mu_pred = np.exp(log_mu + 0.5 * sig_o ** 2) - 1.0
    else:
        mu_pred = np.exp(log_mu).clip(1e-6)
    tmp = df[['team_name', 'conference', 'points_scored']].copy()
    tmp['mu_pred'] = mu_pred
    team_summ = (tmp.groupby('team_name')
                 .agg(obs=('points_scored', 'mean'), pred=('mu_pred', 'mean'))
                 .assign(diff=lambda x: x['pred'] - x['obs']))
    conf_summ = (tmp.groupby('conference')
                 .agg(obs=('points_scored', 'mean'), pred=('mu_pred', 'mean'))
                 .assign(diff=lambda x: x['pred'] - x['obs']))
    n_team_pass = int((team_summ['diff'].abs() <= 2.0).sum())
    n_conf_pass = int((conf_summ['diff'].abs() <= 3.0).sum())
    worst_team  = float(team_summ['diff'].abs().max())
    worst_conf  = float(conf_summ['diff'].abs().max())
    return n_team_pass, n_conf_pass, worst_team, worst_conf


tp_a, cp_a, wt_a, wc_a = mean_checks(samp_a, is_lognormal=False)
tp_b, cp_b, wt_b, wc_b = mean_checks(samp_b, is_lognormal=False)
tp_c, cp_c, wt_c, wc_c = mean_checks(samp_c, is_lognormal=True)
tp_d, cp_d, wt_d, wc_d = mean_checks(samp_d, is_lognormal=True)

# sigma_attack posterior mean
sa_a = float(np.array(samp_a['sigma_attack']).mean())
sa_b = float(np.array(samp_b['sigma_attack']).mean())
sa_c = float(np.array(samp_c['sigma_attack']).mean())
sa_d = float(np.array(samp_d['sigma_attack']).mean())

# variance parameter posterior mean (r mean for A/B, sigma_obs mean for C/D)
def r_mean_str(samp_):
    return f"{float(np.array(samp_['r_negbinom']).mean()):.2f}"
def sig_mean_str(samp_):
    return f"{float(np.array(samp_['sigma_obs']).mean()):.4f}"

# ---------------------------------------------------------------------------
# Final comparison table
# ---------------------------------------------------------------------------
print("=" * 76)
print("DIAGNOSTIC COMPARISON — Models A / B / C / D")
print("(1 chain, 1000 warmup, 1000 samples — R-hat undefined for 1 chain)")
print("=" * 76)
print(f"  {'Metric':<42s}  {'A':>7s}  {'B':>7s}  {'C':>7s}  {'D':>7s}")
print("  " + "-" * 66)
print(f"  {'Likelihood':<42s}  {'NegBin':>7s}  {'NegBin':>7s}  {'LogNrm':>7s}  {'LogNrm':>7s}")
print(f"  {'Variance structure':<42s}  {'Indep':>7s}  {'Hier-r':>7s}  {'Indep':>7s}  {'Hier-s':>7s}")
print(f"  {'Fit time (s)':<42s}  {t_a:>7.0f}  {t_b:>7.0f}  {t_c:>7.0f}  {t_d:>7.0f}")
print(f"  {'Divergences':<42s}  {ndiv_a:>7d}  {ndiv_b:>7d}  {ndiv_c:>7d}  {ndiv_d:>7d}")
print(f"  {'Min ESS_bulk (8 scalar params)':<42s}  {ess_a:>7.0f}  {ess_b:>7.0f}  {ess_c:>7.0f}  {ess_d:>7.0f}")
print(f"  {'sigma_attack posterior mean':<42s}  {sa_a:>7.4f}  {sa_b:>7.4f}  {sa_c:>7.4f}  {sa_d:>7.4f}")
print(f"  {'Variance param mean (r or sigma_obs)':<42s}  {r_mean_str(samp_a):>7s}  {r_mean_str(samp_b):>7s}  {sig_mean_str(samp_c):>7s}  {sig_mean_str(samp_d):>7s}")
print(f"  {'Variance check: confs passed / 10':<42s}  {vpass_a:>7d}  {vpass_b:>7d}  {vpass_c:>7d}  {vpass_d:>7d}")
print(f"  {'Team mean check: teams passed / 131':<42s}  {tp_a:>7d}  {tp_b:>7d}  {tp_c:>7d}  {tp_d:>7d}")
print(f"  {'Team mean check: worst gap (pts)':<42s}  {wt_a:>7.2f}  {wt_b:>7.2f}  {wt_c:>7.2f}  {wt_d:>7.2f}")
print(f"  {'Conf mean check: confs passed / 10':<42s}  {cp_a:>7d}  {cp_b:>7d}  {cp_c:>7d}  {cp_d:>7d}")
print(f"  {'Conf mean check: worst gap (pts)':<42s}  {wc_a:>7.2f}  {wc_b:>7.2f}  {wc_c:>7.2f}  {wc_d:>7.2f}")
print("=" * 76)
print()
print("Cell 10 complete.")

Running diagnostic fits (1 chain, 1000 warmup, 1000 samples)
  Fitting Model A... done (159s)  divergences=0
  Fitting Model B... done (173s)  divergences=0
  Fitting Model C... done (77s)  divergences=0
  Fitting Model D... done (82s)  divergences=0

Running variance checks...
done.

DIAGNOSTIC COMPARISON — Models A / B / C / D
(1 chain, 1000 warmup, 1000 samples — R-hat undefined for 1 chain)
  Metric                                            A        B        C        D
  ------------------------------------------------------------------
  Likelihood                                   NegBin   NegBin   LogNrm   LogNrm
  Variance structure                            Indep   Hier-r    Indep   Hier-s
  Fit time (s)                                    159      173       77       82
  Divergences                                       0        0        0        0
  Min ESS_bulk (8 scalar params)                  179      146      237      193
  sigma_attack posterior mean                  

In [11]:
# =============================================================================
# CELL 11 — Targeted diagnostics on model_08 and diagnostic fit samples
#
# Three checks:
#   Check 1: sigma_attack posterior shape — all four models
#   Check 2: alpha_team posterior means — top/bottom 10 teams by obs scoring
#             (Model C — Log-Normal independent)
#   Check 3: sigma_obs per conference vs observed log-variance (Model C)
#
# No new fits. Uses samp_a, samp_b, samp_c, samp_d from Cell 10
# and samples from the full model_08 fit (Cell 6).
# =============================================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

DIAG_PATH = os.path.join(ARTIFACTS_DIR, "model_08_cell11_diagnostics.png")

# ── observed team means (needed for Check 2) ──────────────────────────────
team_obs_mean = (
    df.groupby('team_name')['points_scored']
    .mean()
    .sort_values(ascending=False)
)
top10_teams    = team_obs_mean.head(10).index.tolist()
bottom10_teams = team_obs_mean.tail(10).index.tolist()

# ── CHECK 1 — sigma_attack posterior shape, all four models ───────────────
print("=" * 68)
print("CHECK 1 — sigma_attack posterior summary (1000 draws, 1 chain)")
print("=" * 68)
print(f"  {'Model':<10s}  {'Mean':>8s}  {'Std':>8s}  {'5th':>8s}  "
      f"{'25th':>8s}  {'Median':>8s}  {'75th':>8s}  {'95th':>8s}  "
      f"{'Pct<0.05':>9s}")
print("  " + "-" * 82)

model_labels  = ['A (NB indep)', 'B (NB hier)', 'C (LN indep)', 'D (LN hier)']
model_samples = [samp_a, samp_b, samp_c, samp_d]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, (label, samp_) in enumerate(zip(model_labels, model_samples)):
    sa = np.array(samp_['sigma_attack'])
    p5, p25, p50, p75, p95 = np.percentile(sa, [5, 25, 50, 75, 95])
    pct_near_zero = float((sa < 0.05).mean()) * 100
    print(f"  {label:<12s}  {sa.mean():>8.4f}  {sa.std():>8.4f}  "
          f"{p5:>8.4f}  {p25:>8.4f}  {p50:>8.4f}  {p75:>8.4f}  "
          f"{p95:>8.4f}  {pct_near_zero:>8.1f}%")

    ax = axes[i]
    ax.hist(sa, bins=50, color='steelblue', alpha=0.75, density=True)
    ax.axvline(sa.mean(), color='red',    lw=1.5, label=f'mean={sa.mean():.3f}')
    ax.axvline(p50,       color='orange', lw=1.5, linestyle='--',
               label=f'median={p50:.3f}')
    ax.set_title(f'Model {label}\nsigma_attack', fontsize=9)
    ax.set_xlabel('sigma_attack')
    ax.legend(fontsize=7)

print()

# Also print model_08 full fit sigma_attack for reference
sa_08 = np.array(samples['sigma_attack'])
p5, p25, p50, p75, p95 = np.percentile(sa_08, [5, 25, 50, 75, 95])
pct_near_zero = float((sa_08 < 0.05).mean()) * 100
print(f"  {'model_08 (4ch)':<12s}  {sa_08.mean():>8.4f}  {sa_08.std():>8.4f}  "
      f"{p5:>8.4f}  {p25:>8.4f}  {p50:>8.4f}  {p75:>8.4f}  "
      f"{p95:>8.4f}  {pct_near_zero:>8.1f}%")
print()

# ── CHECK 2 — alpha_team posterior means, Model C, top/bottom 10 ──────────
print("=" * 68)
print("CHECK 2 — alpha_team posterior means, Model C (Log-Normal independent)")
print("          Top 10 and bottom 10 teams by observed scoring mean")
print("=" * 68)

alpha_c    = np.array(samp_c['alpha_team'])   # (1000, 131)
alpha_mean = alpha_c.mean(axis=0)             # (131,)
alpha_std  = alpha_c.std(axis=0)              # (131,)

# mu_pred per team: exp(mu_league + conf + alpha - delta + ...) averaged
# Use posterior mean of all params for a clean comparison
mu_l_c   = float(np.array(samp_c['mu_league']).mean())
hfa_l_c  = float(np.array(samp_c['hfa_league']).mean())
mu_c_arr = np.array(samp_c['mu_conference']).mean(axis=0)   # (10,)
alp_c    = np.array(samp_c['alpha_team']).mean(axis=0)      # (131,)
dlt_c    = np.array(samp_c['delta_team']).mean(axis=0)      # (131,)
hfa_c    = np.array(samp_c['hfa_team']).mean(axis=0)        # (131,)
sig_c    = np.array(samp_c['sigma_obs']).mean(axis=0)       # (10,)

log_mu_c = (
    mu_l_c
    + mu_c_arr[conf_idx_arr]
    + alp_c[team_idx_arr]
    - dlt_c[opp_idx_arr]
    + hfa_l_c * is_home_arr
    + hfa_c[team_idx_arr] * is_home_arr
)
sig_o_c  = sig_c[conf_idx_arr]
# Log-Normal mean: exp(mu + 0.5*sigma^2) - 1
mu_pred_c = np.exp(log_mu_c + 0.5 * sig_o_c**2) - 1.0

tmp_c = df[['team_name', 'points_scored']].copy()
tmp_c['mu_pred'] = mu_pred_c
team_pred_c = (tmp_c.groupby('team_name')
               .agg(obs=('points_scored', 'mean'),
                    pred=('mu_pred', 'mean'))
               .assign(diff=lambda x: x['pred'] - x['obs']))

print(f"\n  TOP 10 teams by observed scoring mean:")
print(f"  {'Team':<28s}  {'Obs Mean':>9s}  {'Pred Mean':>9s}  {'Diff':>7s}  "
      f"{'alpha_mean':>10s}  {'alpha_std':>9s}")
print("  " + "-" * 78)
for team in top10_teams:
    row   = team_pred_c.loc[team]
    t_idx = team_to_idx[team]
    print(f"  {team:<28s}  {row['obs']:>9.2f}  {row['pred']:>9.2f}  "
          f"{row['diff']:>+7.2f}  {alpha_mean[t_idx]:>10.4f}  "
          f"{alpha_std[t_idx]:>9.4f}")

print(f"\n  BOTTOM 10 teams by observed scoring mean:")
print(f"  {'Team':<28s}  {'Obs Mean':>9s}  {'Pred Mean':>9s}  {'Diff':>7s}  "
      f"{'alpha_mean':>10s}  {'alpha_std':>9s}")
print("  " + "-" * 78)
for team in bottom10_teams:
    row   = team_pred_c.loc[team]
    t_idx = team_to_idx[team]
    print(f"  {team:<28s}  {row['obs']:>9.2f}  {row['pred']:>9.2f}  "
          f"{row['diff']:>+7.2f}  {alpha_mean[t_idx]:>10.4f}  "
          f"{alpha_std[t_idx]:>9.4f}")

# What alpha_team would Oregon need to be predicted correctly?
oregon_idx  = team_to_idx['Oregon']
oregon_obs  = team_pred_c.loc['Oregon', 'obs']
oregon_pred_no_alpha = float(np.exp(
    mu_l_c
    + mu_c_arr[conf_to_idx['Pac-12']]
    - dlt_c[oregon_idx]          # opponent effect is zero for self-prediction
    + 0.5 * sig_c[conf_to_idx['Pac-12']]**2
) - 1.0)

# Required log_mu for Oregon to hit obs mean (ignoring opponent and hfa)
required_log_mu = np.log(oregon_obs + 1.0) - 0.5 * sig_c[conf_to_idx['Pac-12']]**2
required_alpha  = required_log_mu - (mu_l_c + mu_c_arr[conf_to_idx['Pac-12']])
required_alpha_raw = required_alpha / float(np.array(samp_c['sigma_attack']).mean())

print(f"\n  Oregon implied alpha_team needed  : {required_alpha:.4f}")
print(f"  Oregon sigma_attack posterior mean: {float(np.array(samp_c['sigma_attack']).mean()):.4f}")
print(f"  Implied alpha_team_raw needed     : {required_alpha_raw:.2f}  "
      f"(Normal(0,1) prior — values > 3 are essentially impossible)")
print()

# ── CHECK 3 — sigma_obs per conference vs observed log-variance, Model C ──
print("=" * 68)
print("CHECK 3 — sigma_obs per conference vs observed log-variance (Model C)")
print("=" * 68)
sig_obs_draws = np.array(samp_c['sigma_obs'])   # (1000, 10)
print(f"\n  {'Conference':<22s}  {'Obs log-var':>11s}  {'Obs log-std':>11s}  "
      f"{'sigma_obs mean':>14s}  {'sigma_obs 5%':>12s}  {'sigma_obs 95%':>13s}  "
      f"{'sigma²_obs mean':>15s}  {'Gap (obs-model)':>15s}")
print("  " + "-" * 115)
for i, c in enumerate(CONFERENCES):
    obs_lv  = obs_log_var[c]
    obs_ls  = float(np.sqrt(obs_lv))
    s_mean  = float(sig_obs_draws[:, i].mean())
    s_p5    = float(np.percentile(sig_obs_draws[:, i], 5))
    s_p95   = float(np.percentile(sig_obs_draws[:, i], 95))
    s2_mean = s_mean ** 2
    gap     = obs_lv - s2_mean
    print(f"  {c:<22s}  {obs_lv:>11.4f}  {obs_ls:>11.4f}  "
          f"{s_mean:>14.4f}  {s_p5:>12.4f}  {s_p95:>13.4f}  "
          f"{s2_mean:>15.4f}  {gap:>+15.4f}")

print()
print("  Note: sigma_obs captures residual variance AFTER mean structure.")
print("  If gap is large and positive, mean structure is not capturing")
print("  team-level spread — residual is larger than sigma_obs can express.")
print("  If sigma_obs^2 < obs_log_var systematically, the mean structure")
print("  is under-fitting team effects and leaving variance on the table.")
print()

# ── Save figure ────────────────────────────────────────────────────────────
plt.suptitle("Cell 11 — sigma_attack posteriors (all four diagnostic models)",
             fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig(DIAG_PATH, dpi=120, bbox_inches='tight')
plt.close()
print(f"Plot saved : {DIAG_PATH}")
print()
print("Cell 11 complete.")

CHECK 1 — sigma_attack posterior summary (1000 draws, 1 chain)
  Model           Mean       Std       5th      25th    Median      75th      95th   Pct<0.05
  ----------------------------------------------------------------------------------
  A (NB indep)    0.0259    0.0121    0.0049    0.0170    0.0265    0.0345    0.0454      98.5%
  B (NB hier)     0.0297    0.0129    0.0058    0.0206    0.0304    0.0392    0.0492      96.0%
  C (LN indep)    0.0271    0.0147    0.0044    0.0154    0.0275    0.0375    0.0525      93.4%
  D (LN hier)     0.0266    0.0155    0.0029    0.0134    0.0267    0.0380    0.0520      93.2%

  model_08 (4ch)    0.0239    0.0131    0.0030    0.0138    0.0240    0.0335    0.0458      97.8%

CHECK 2 — alpha_team posterior means, Model C (Log-Normal independent)
          Top 10 and bottom 10 teams by observed scoring mean

  TOP 10 teams by observed scoring mean:
  Team                           Obs Mean  Pred Mean     Diff  alpha_mean  alpha_std
  ------------

In [12]:
# =============================================================================
# CELL 12 — Three-way model comparison: Options 1, 2, 3
#
# All three use Log-Normal likelihood (Model C architecture).
# Difference is in how team effects and features interact.
#
# Option 1 — No team random effects. Pure feature model.
#             alpha_team, delta_team, sigma_attack, sigma_defense removed.
#             Features carry all team-quality signal.
#
# Option 2 — Team random effects kept. Team-quality features zeroed out.
#             sp_rating, pregame_elo, last3_win_pct,
#             last3_pts_scored_avg, last3_pts_allowed_avg set to zero.
#             Team effects carry all team-quality signal.
#
# Option 3 — Team random effects kept. Team-quality features centered
#             at each team's own season mean. Features carry within-team
#             variation; team effects carry between-team variation.
#             Clean orthogonalization — both components do their job.
#
# Diagnostic fit: 1 chain, 1000 warmup, 1000 samples, PRNGKey(0).
# Log-Normal likelihood throughout. GameData reused from Cell 5.
# =============================================================================

import numpy as np
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value
from jax import random
import arviz as az
import time

# ---------------------------------------------------------------------------
# Data preparation
# ---------------------------------------------------------------------------

# Shared log-points target
log_points = jnp.array(
    np.log1p(np.array(df['points_scored'].values, dtype=np.float32)),
    dtype=jnp.float32
)

conf_labels   = np.array(df['conference'].values)
log_points_np = np.array(log_points)
points_np     = np.array(df['points_scored'].values, dtype=np.float32)

obs_log_var = {}
obs_vmr     = {}
for c_name in CONFERENCES:
    mask  = conf_labels == c_name
    y_log = log_points_np[mask]
    y_raw = points_np[mask]
    obs_log_var[c_name] = float(y_log.var())
    mean_c = y_raw.mean()
    if mean_c > 1e-6:
        obs_vmr[c_name] = float(y_raw.var() / mean_c)

team_idx_arr = np.array(team_idx,  dtype=np.int32)
opp_idx_arr  = np.array(opp_idx,   dtype=np.int32)
conf_idx_arr = np.array(conf_idx,  dtype=np.int32)
is_home_arr  = np.array(df['is_home'].values, dtype=np.float32)

# ---------------------------------------------------------------------------
# Option 3 — center team-quality features at team-season mean
#
# Features centered: sp_rating, pregame_elo, last3_win_pct,
#                    last3_points_scored_avg, last3_points_allowed_avg
#
# For each feature, compute mean per (team_name, season), then subtract.
# This leaves within-team-season variation only.
# Between-team variation is absorbed by alpha_team / delta_team.
# ---------------------------------------------------------------------------
team_quality_features = [
    'sp_rating',
    'pregame_elo',
    'last3_win_pct',
    'last3_points_scored_avg',
    'last3_points_allowed_avg',
]

df_opt3 = df.copy()
for feat in team_quality_features:
    team_season_mean = (
        df_opt3.groupby(['team_name', 'season'])[feat]
        .transform('mean')
    )
    df_opt3[feat] = df_opt3[feat] - team_season_mean

print("Option 3 — centered feature stats (expect mean≈0 within team-season):")
print(f"  {'Feature':<30s}  {'Global mean':>12s}  {'Global std':>10s}  "
      f"{'Min':>8s}  {'Max':>8s}")
for feat in team_quality_features:
    vals = df_opt3[feat].values
    print(f"  {feat:<30s}  {vals.mean():>12.4f}  {vals.std():>10.4f}  "
          f"{vals.min():>8.4f}  {vals.max():>8.4f}")
print()

# Option 2 — zero out team-quality features in a copy of training_data
import dataclasses

def zero_feature(training_data, field_name):
    arr = getattr(training_data, field_name)
    return jnp.zeros_like(arr)

training_data_opt2 = dataclasses.replace(
    training_data,
    sp_rating          = zero_feature(training_data, 'sp_rating'),
    pregame_elo        = zero_feature(training_data, 'pregame_elo'),
    last3_win_pct      = zero_feature(training_data, 'last3_win_pct'),
    last3_pts_scored_avg  = zero_feature(training_data, 'last3_pts_scored_avg'),
    last3_pts_allowed_avg = zero_feature(training_data, 'last3_pts_allowed_avg'),
)

# Option 3 — rebuild GameData with centered features
def to_f32_opt3(col):
    return jnp.array(df_opt3[col].values, dtype=jnp.float32)

training_data_opt3 = dataclasses.replace(
    training_data,
    sp_rating             = to_f32_opt3('sp_rating'),
    pregame_elo           = to_f32_opt3('pregame_elo'),
    last3_win_pct         = to_f32_opt3('last3_win_pct'),
    last3_pts_scored_avg  = to_f32_opt3('last3_points_scored_avg'),
    last3_pts_allowed_avg = to_f32_opt3('last3_points_allowed_avg'),
)

print("Option 2 — zeroed feature check (expect all zeros):")
for fname in ['sp_rating', 'pregame_elo', 'last3_win_pct']:
    arr = getattr(training_data_opt2, fname)
    print(f"  {fname:<30s}  sum={float(arr.sum()):.4f}")
print()

print("Option 3 — centered feature check (expect mean≈0):")
for fname, col in [('sp_rating', 'sp_rating'),
                   ('pregame_elo', 'pregame_elo'),
                   ('last3_win_pct', 'last3_win_pct')]:
    arr = getattr(training_data_opt3, fname)
    print(f"  {fname:<30s}  mean={float(arr.mean()):.6f}  std={float(arr.std()):.4f}")
print()

# ---------------------------------------------------------------------------
# Model definitions
# ---------------------------------------------------------------------------

# Common linear predictor helper — same for all three options
def _compute_log_mu(data, mu_league, hfa_league, mu_conference,
                    alpha_team, delta_team, hfa_team,
                    sp_weight, rec_weight,
                    b_close_game_epa, b_close_game_def_epa,
                    b_pregame_elo, b_elo_sp_divergence, b_last3_win_pct,
                    b_away_travel_distance, b_away_tz_delta, b_wind_chill,
                    b_rush_rate_std_downs, b_rush_rate_pass_downs,
                    b_off_sack_rate_allowed, b_def_sack_rate,
                    b_off_archetype, b_def_archetype,
                    b_last3_off_epa_avg, b_last3_def_epa_avg,
                    b_last3_pts_scored_avg, b_last3_pts_allowed_avg,
                    b_days_since_last_game, b_close_play_count_delta,
                    b_away_elevation_delta):
    return (
        mu_league
        + mu_conference[data.conf_idx]
        + alpha_team[data.team_idx]
        - delta_team[data.opp_idx]
        + (hfa_league + hfa_team[data.team_idx]) * data.is_home
        + sp_weight              * data.sp_rating
        + rec_weight[data.conf_idx] * data.recruiting_3yr_avg
        + b_close_game_epa       * data.close_game_epa
        + b_close_game_def_epa   * data.close_game_def_epa
        + b_pregame_elo          * data.pregame_elo
        + b_elo_sp_divergence    * data.elo_sp_divergence
        + b_last3_win_pct        * data.last3_win_pct
        + b_away_travel_distance * data.away_travel_distance
        + b_away_tz_delta        * data.away_tz_delta
        + b_wind_chill           * data.wind_chill
        + b_rush_rate_std_downs  * data.rush_rate_std_downs
        + b_rush_rate_pass_downs * data.rush_rate_pass_downs
        + b_off_sack_rate_allowed * data.off_sack_rate_allowed
        + b_def_sack_rate         * data.def_sack_rate
        + b_off_archetype[data.off_archetype_idx]
        + b_def_archetype[data.def_archetype_idx]
        + b_last3_off_epa_avg     * data.mask_last3_off_epa    * data.last3_off_epa_avg
        + b_last3_def_epa_avg     * data.mask_last3_def_epa    * data.last3_def_epa_avg
        + b_last3_pts_scored_avg  * data.mask_last3_pts_scored * data.last3_pts_scored_avg
        + b_last3_pts_allowed_avg * data.mask_last3_pts_allowed * data.last3_pts_allowed_avg
        + b_days_since_last_game  * data.mask_days_since        * data.days_since_last_game
        + b_close_play_count_delta * data.mask_close_play_count * data.close_play_count_delta
        + b_away_elevation_delta  * data.mask_elevation         * data.away_elevation_delta
    )


def _sample_coefs():
    """Sample all game-level and prior-seed coefficients."""
    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 0.15))
    rec_weight_other   = numpyro.sample(
        "rec_weight_other", dist.Normal(0.0, 0.5),
        sample_shape=(N_CONFERENCES - 1,)
    )
    rec_weight_sunbelt = numpyro.sample(
        "rec_weight_sunbelt", dist.TruncatedNormal(0.0, 0.5, high=0.0)
    )
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])
    b_close_game_epa       = numpyro.sample("b_close_game_epa",       dist.Normal(0.0, 0.10))
    b_close_game_def_epa   = numpyro.sample("b_close_game_def_epa",   dist.Normal(0.0, 0.10))
    b_pregame_elo          = numpyro.sample("b_pregame_elo",          dist.Normal(0.0, 0.15))
    b_elo_sp_divergence    = numpyro.sample("b_elo_sp_divergence",    dist.Normal(0.0, 0.15))
    b_last3_win_pct        = numpyro.sample("b_last3_win_pct",        dist.Normal(0.0, 0.15))
    b_away_travel_distance = numpyro.sample("b_away_travel_distance", dist.Normal(0.0, 0.15))
    b_away_tz_delta        = numpyro.sample("b_away_tz_delta",        dist.Normal(0.0, 0.15))
    b_wind_chill           = numpyro.sample("b_wind_chill",           dist.Normal(0.0, 0.15))
    b_rush_rate_std_downs  = numpyro.sample("b_rush_rate_std_downs",  dist.Normal(0.0, 0.15))
    b_rush_rate_pass_downs = numpyro.sample("b_rush_rate_pass_downs", dist.Normal(0.0, 0.15))
    b_off_sack_rate_allowed= numpyro.sample("b_off_sack_rate_allowed",dist.Normal(0.0, 0.15))
    b_def_sack_rate        = numpyro.sample("b_def_sack_rate",        dist.Normal(0.0, 0.15))
    b_off_archetype = numpyro.sample(
        "b_off_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_def_archetype = numpyro.sample(
        "b_def_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_last3_off_epa_avg      = numpyro.sample("b_last3_off_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_def_epa_avg      = numpyro.sample("b_last3_def_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_pts_scored_avg   = numpyro.sample("b_last3_pts_scored_avg",   dist.Normal(0.0, 0.15))
    b_last3_pts_allowed_avg  = numpyro.sample("b_last3_pts_allowed_avg",  dist.Normal(0.0, 0.15))
    b_days_since_last_game   = numpyro.sample("b_days_since_last_game",   dist.Normal(0.0, 0.15))
    b_close_play_count_delta = numpyro.sample("b_close_play_count_delta", dist.Normal(0.0, 0.15))
    b_away_elevation_delta   = numpyro.sample("b_away_elevation_delta",   dist.Normal(0.0, 0.15))
    return (sp_weight, rec_weight,
            b_close_game_epa, b_close_game_def_epa, b_pregame_elo,
            b_elo_sp_divergence, b_last3_win_pct, b_away_travel_distance,
            b_away_tz_delta, b_wind_chill, b_rush_rate_std_downs,
            b_rush_rate_pass_downs, b_off_sack_rate_allowed, b_def_sack_rate,
            b_off_archetype, b_def_archetype, b_last3_off_epa_avg,
            b_last3_def_epa_avg, b_last3_pts_scored_avg, b_last3_pts_allowed_avg,
            b_days_since_last_game, b_close_play_count_delta, b_away_elevation_delta)


# OPTION 1 — No team random effects
def model_opt1(data: GameData, N_teams: int, log_points: jnp.ndarray):
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))

    sigma_obs = numpyro.sample(
        "sigma_obs", dist.HalfNormal(0.3), sample_shape=(N_CONFERENCES,)
    )
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(0.1))
    mu_conference    = numpyro.sample(
        "mu_conference", dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )

    # No alpha_team, delta_team, sigma_attack, sigma_defense
    # Replaced with zero vectors — same _compute_log_mu signature
    alpha_team = jnp.zeros(N_teams)
    delta_team = jnp.zeros(N_teams)

    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team_raw   = numpyro.sample(
        "hfa_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    hfa_team = numpyro.deterministic("hfa_team", hfa_team_raw * sigma_hfa_team)

    coefs = _sample_coefs()
    log_mu = _compute_log_mu(data, mu_league, hfa_league, mu_conference,
                             alpha_team, delta_team, hfa_team, *coefs)
    numpyro.sample("obs",
                   dist.Normal(log_mu, sigma_obs[data.conf_idx]),
                   obs=log_points)


# OPTION 2 — Team effects kept, team-quality features zeroed in data
def model_opt2(data: GameData, N_teams: int, log_points: jnp.ndarray):
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))

    sigma_obs = numpyro.sample(
        "sigma_obs", dist.HalfNormal(0.3), sample_shape=(N_CONFERENCES,)
    )
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(0.1))
    mu_conference    = numpyro.sample(
        "mu_conference", dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )

    sigma_attack   = numpyro.sample("sigma_attack",   dist.HalfNormal(0.25))
    alpha_team_raw = numpyro.sample(
        "alpha_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    alpha_team = numpyro.deterministic("alpha_team", alpha_team_raw * sigma_attack)

    sigma_defense  = numpyro.sample("sigma_defense",  dist.HalfNormal(0.25))
    delta_team_raw = numpyro.sample(
        "delta_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    delta_team = numpyro.deterministic("delta_team", delta_team_raw * sigma_defense)

    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team_raw   = numpyro.sample(
        "hfa_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    hfa_team = numpyro.deterministic("hfa_team", hfa_team_raw * sigma_hfa_team)

    coefs = _sample_coefs()
    log_mu = _compute_log_mu(data, mu_league, hfa_league, mu_conference,
                             alpha_team, delta_team, hfa_team, *coefs)
    numpyro.sample("obs",
                   dist.Normal(log_mu, sigma_obs[data.conf_idx]),
                   obs=log_points)


# OPTION 3 — Team effects kept, team-quality features centered at team-season mean
# Model function identical to Option 2 — difference is in the data passed in
def model_opt3(data: GameData, N_teams: int, log_points: jnp.ndarray):
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))

    sigma_obs = numpyro.sample(
        "sigma_obs", dist.HalfNormal(0.3), sample_shape=(N_CONFERENCES,)
    )
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(0.1))
    mu_conference    = numpyro.sample(
        "mu_conference", dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )

    sigma_attack   = numpyro.sample("sigma_attack",   dist.HalfNormal(0.25))
    alpha_team_raw = numpyro.sample(
        "alpha_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    alpha_team = numpyro.deterministic("alpha_team", alpha_team_raw * sigma_attack)

    sigma_defense  = numpyro.sample("sigma_defense",  dist.HalfNormal(0.25))
    delta_team_raw = numpyro.sample(
        "delta_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    delta_team = numpyro.deterministic("delta_team", delta_team_raw * sigma_defense)

    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team_raw   = numpyro.sample(
        "hfa_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    hfa_team = numpyro.deterministic("hfa_team", hfa_team_raw * sigma_hfa_team)

    coefs = _sample_coefs()
    log_mu = _compute_log_mu(data, mu_league, hfa_league, mu_conference,
                             alpha_team, delta_team, hfa_team, *coefs)
    numpyro.sample("obs",
                   dist.Normal(log_mu, sigma_obs[data.conf_idx]),
                   obs=log_points)


# ---------------------------------------------------------------------------
# Init values
# ---------------------------------------------------------------------------
base_init = {
    "mu_league"          : jnp.array(3.3),
    "hfa_league"         : jnp.array(0.1),
    "sigma_obs"          : jnp.ones(N_CONFERENCES) * 0.3,
    "sigma_conference"   : jnp.array(0.05),
    "mu_conference"      : jnp.zeros(N_CONFERENCES),
    "sigma_hfa_team"     : jnp.array(0.05),
    "hfa_team_raw"       : jnp.zeros(N_teams),
    "sp_weight"          : jnp.array(0.0),
    "rec_weight_other"   : jnp.zeros(N_CONFERENCES - 1),
    "rec_weight_sunbelt" : jnp.array(-0.1),
    "b_close_game_epa"        : jnp.array(0.0),
    "b_close_game_def_epa"    : jnp.array(0.0),
    "b_pregame_elo"           : jnp.array(0.0),
    "b_elo_sp_divergence"     : jnp.array(0.0),
    "b_last3_win_pct"         : jnp.array(0.0),
    "b_away_travel_distance"  : jnp.array(0.0),
    "b_away_tz_delta"         : jnp.array(0.0),
    "b_wind_chill"            : jnp.array(0.0),
    "b_rush_rate_std_downs"   : jnp.array(0.0),
    "b_rush_rate_pass_downs"  : jnp.array(0.0),
    "b_off_sack_rate_allowed" : jnp.array(0.0),
    "b_def_sack_rate"         : jnp.array(0.0),
    "b_off_archetype"         : jnp.zeros(4),
    "b_def_archetype"         : jnp.zeros(4),
    "b_last3_off_epa_avg"     : jnp.array(0.0),
    "b_last3_def_epa_avg"     : jnp.array(0.0),
    "b_last3_pts_scored_avg"  : jnp.array(0.0),
    "b_last3_pts_allowed_avg" : jnp.array(0.0),
    "b_days_since_last_game"  : jnp.array(0.0),
    "b_close_play_count_delta": jnp.array(0.0),
    "b_away_elevation_delta"  : jnp.array(0.0),
}

team_effect_init = {
    "sigma_attack"  : jnp.array(0.05),
    "alpha_team_raw": jnp.zeros(N_teams),
    "sigma_defense" : jnp.array(0.05),
    "delta_team_raw": jnp.zeros(N_teams),
}

init_opt1 = {**base_init}                         # no team effect keys
init_opt2 = {**base_init, **team_effect_init}
init_opt3 = {**base_init, **team_effect_init}


# ---------------------------------------------------------------------------
# Fit function — 1 chain, 1000 warmup, 1000 samples
# ---------------------------------------------------------------------------
def run_diagnostic(label, model_fn, init_vals, model_kwargs):
    print(f"  Fitting Option {label}...", end=" ", flush=True)
    nuts  = NUTS(model_fn, target_accept_prob=0.9,
                 init_strategy=init_to_value(values=init_vals))
    mcmc_ = MCMC(nuts, num_warmup=1000, num_samples=1000,
                 num_chains=1, progress_bar=False)
    numpyro.enable_validation(False)
    try:
        t0 = time.time()
        mcmc_.run(random.PRNGKey(0), **model_kwargs)
        elapsed = time.time() - t0
    finally:
        numpyro.enable_validation(True)
    extra_ = mcmc_.get_extra_fields()
    n_div  = int(extra_['diverging'].sum()) if 'diverging' in extra_ else 0
    samp_  = mcmc_.get_samples()
    idata_ = az.from_numpyro(mcmc_)
    print(f"done ({elapsed:.0f}s)  divergences={n_div}")
    return samp_, idata_, n_div, elapsed


print("=" * 60)
print("Running diagnostic fits (1 chain, 1000 warmup, 1000 samples)")
print("=" * 60)

samp_1, idata_1, ndiv_1, t_1 = run_diagnostic(
    "1 (no team effects)", model_opt1, init_opt1,
    {"data": training_data,      "N_teams": N_teams, "log_points": log_points}
)
samp_2, idata_2, ndiv_2, t_2 = run_diagnostic(
    "2 (zeroed features)", model_opt2, init_opt2,
    {"data": training_data_opt2, "N_teams": N_teams, "log_points": log_points}
)
samp_3, idata_3, ndiv_3, t_3 = run_diagnostic(
    "3 (centered features)", model_opt3, init_opt3,
    {"data": training_data_opt3, "N_teams": N_teams, "log_points": log_points}
)
print()


# ---------------------------------------------------------------------------
# ESS_bulk via az.ess (single-chain compatible)
# ---------------------------------------------------------------------------
def get_min_ess(idata_, has_team_effects=True):
    scalar_vars = ["mu_league", "hfa_league", "sigma_conference",
                   "sigma_hfa_team", "sp_weight", "b_close_game_epa"]
    if has_team_effects:
        scalar_vars += ["sigma_attack", "sigma_defense"]
    ess = az.ess(idata_, var_names=scalar_vars)
    return min(float(ess[v].values.min()) for v in scalar_vars if v in ess)

ess_1 = get_min_ess(idata_1, has_team_effects=False)
ess_2 = get_min_ess(idata_2, has_team_effects=True)
ess_3 = get_min_ess(idata_3, has_team_effects=True)


# ---------------------------------------------------------------------------
# sigma_attack posterior summary (Options 2 and 3 only)
# ---------------------------------------------------------------------------
def sigma_attack_summary(samp_):
    sa = np.array(samp_['sigma_attack'])
    return float(sa.mean()), float(np.percentile(sa, 50)), float(sa.std())

sa2_mean, sa2_med, sa2_std = sigma_attack_summary(samp_2)
sa3_mean, sa3_med, sa3_std = sigma_attack_summary(samp_3)


# ---------------------------------------------------------------------------
# Log-variance check — 100 draws per model
# ---------------------------------------------------------------------------
def logvar_check(samp_, data_arr, has_team_effects=True):
    mu_l  = np.array(samp_['mu_league'])
    hfa_l = np.array(samp_['hfa_league'])
    mu_c  = np.array(samp_['mu_conference'])
    hfa   = np.array(samp_['hfa_team'])
    sig   = np.array(samp_['sigma_obs'])
    if has_team_effects:
        alp = np.array(samp_['alpha_team'])
        dlt = np.array(samp_['delta_team'])
    else:
        alp = np.zeros((len(mu_l), N_teams))
        dlt = np.zeros((len(mu_l), N_teams))
    rng   = np.random.default_rng(0)
    draws = rng.choice(len(mu_l), size=100, replace=False)
    conf_draws = {c: [] for c in CONFERENCES}
    for d in draws:
        log_mu = (mu_l[d] + mu_c[d, conf_idx_arr]
                  + alp[d, team_idx_arr] - dlt[d, opp_idx_arr]
                  + hfa_l[d] * is_home_arr
                  + hfa[d, team_idx_arr] * is_home_arr)
        sig_o  = sig[d, conf_idx_arr]
        y_log  = rng.normal(loc=log_mu, scale=sig_o)
        for c in CONFERENCES:
            mask = conf_labels == c
            conf_draws[c].append(y_log[mask].var())
    n_pass = 0
    for c in CONFERENCES:
        dc = np.array(conf_draws[c])
        lo, hi = np.percentile(dc, 5), np.percentile(dc, 95)
        if lo <= obs_log_var[c] <= hi:
            n_pass += 1
    return n_pass

print("Running log-variance checks...", flush=True)
vpass_1 = logvar_check(samp_1, training_data,      has_team_effects=False)
vpass_2 = logvar_check(samp_2, training_data_opt2, has_team_effects=True)
vpass_3 = logvar_check(samp_3, training_data_opt3, has_team_effects=True)
print("done.")
print()


# ---------------------------------------------------------------------------
# Team-level and conference-level mean checks with full detail
# ---------------------------------------------------------------------------
def mean_checks(samp_, data_for_pred, has_team_effects=True):
    mu_l  = np.array(samp_['mu_league']).mean()
    hfa_l = np.array(samp_['hfa_league']).mean()
    mu_c  = np.array(samp_['mu_conference']).mean(axis=0)
    hfa   = np.array(samp_['hfa_team']).mean(axis=0)
    sig   = np.array(samp_['sigma_obs']).mean(axis=0)
    if has_team_effects:
        alp = np.array(samp_['alpha_team']).mean(axis=0)
        dlt = np.array(samp_['delta_team']).mean(axis=0)
    else:
        alp = np.zeros(N_teams)
        dlt = np.zeros(N_teams)

    # Use the actual data arrays from the GameData passed to this model
    t_idx = np.array(data_for_pred.team_idx, dtype=np.int32)
    o_idx = np.array(data_for_pred.opp_idx,  dtype=np.int32)
    c_idx = np.array(data_for_pred.conf_idx, dtype=np.int32)
    ih    = np.array(data_for_pred.is_home,  dtype=np.float32)

    log_mu  = (mu_l + mu_c[c_idx] + alp[t_idx] - dlt[o_idx]
               + hfa_l * ih + hfa[t_idx] * ih)
    sig_o   = sig[c_idx]
    mu_pred = np.exp(log_mu + 0.5 * sig_o ** 2) - 1.0

    tmp = df[['team_name', 'conference', 'points_scored']].copy()
    tmp['mu_pred'] = mu_pred

    team_summ = (tmp.groupby('team_name')
                 .agg(obs=('points_scored', 'mean'),
                      pred=('mu_pred', 'mean'))
                 .assign(diff=lambda x: x['pred'] - x['obs']))
    conf_summ = (tmp.groupby('conference')
                 .agg(obs=('points_scored', 'mean'),
                      pred=('mu_pred', 'mean'))
                 .assign(diff=lambda x: x['pred'] - x['obs']))

    n_team_pass = int((team_summ['diff'].abs() <= 2.0).sum())
    n_conf_pass = int((conf_summ['diff'].abs() <= 3.0).sum())
    worst_team  = float(team_summ['diff'].abs().max())
    worst_conf  = float(conf_summ['diff'].abs().max())
    return n_team_pass, n_conf_pass, worst_team, worst_conf

tp_1, cp_1, wt_1, wc_1 = mean_checks(samp_1, training_data,      has_team_effects=False)
tp_2, cp_2, wt_2, wc_2 = mean_checks(samp_2, training_data_opt2, has_team_effects=True)
tp_3, cp_3, wt_3, wc_3 = mean_checks(samp_3, training_data_opt3, has_team_effects=True)


# ---------------------------------------------------------------------------
# Alpha_team detail for Options 2 and 3 — top 5 and bottom 5 teams
# ---------------------------------------------------------------------------
def alpha_team_detail(label, samp_, data_for_pred):
    alpha_mean = np.array(samp_['alpha_team']).mean(axis=0)
    sig        = np.array(samp_['sigma_obs']).mean(axis=0)

    mu_l  = np.array(samp_['mu_league']).mean()
    hfa_l = np.array(samp_['hfa_league']).mean()
    mu_c  = np.array(samp_['mu_conference']).mean(axis=0)
    alp   = alpha_mean
    dlt   = np.array(samp_['delta_team']).mean(axis=0)
    hfa   = np.array(samp_['hfa_team']).mean(axis=0)

    t_idx = np.array(data_for_pred.team_idx, dtype=np.int32)
    o_idx = np.array(data_for_pred.opp_idx,  dtype=np.int32)
    c_idx = np.array(data_for_pred.conf_idx, dtype=np.int32)
    ih    = np.array(data_for_pred.is_home,  dtype=np.float32)

    log_mu  = (mu_l + mu_c[c_idx] + alp[t_idx] - dlt[o_idx]
               + hfa_l * ih + hfa[t_idx] * ih)
    sig_o   = sig[c_idx]
    mu_pred = np.exp(log_mu + 0.5 * sig_o**2) - 1.0

    tmp = df[['team_name', 'points_scored']].copy()
    tmp['mu_pred'] = mu_pred
    team_summ = (tmp.groupby('team_name')
                 .agg(obs=('points_scored', 'mean'),
                      pred=('mu_pred', 'mean'))
                 .assign(diff=lambda x: x['pred'] - x['obs']))

    sa_mean = float(np.array(samp_['sigma_attack']).mean())
    print(f"\n  Option {label} — sigma_attack mean={sa_mean:.4f}")
    print(f"  {'Team':<28s}  {'Obs':>7s}  {'Pred':>7s}  {'Diff':>7s}  {'alpha':>8s}")
    print("  " + "-" * 62)
    for team in top10_teams[:5]:
        row = team_summ.loc[team]
        t_i = team_to_idx[team]
        print(f"  {team:<28s}  {row['obs']:>7.2f}  {row['pred']:>7.2f}  "
              f"{row['diff']:>+7.2f}  {alpha_mean[t_i]:>8.4f}")
    print("  ...")
    for team in bottom10_teams[-5:]:
        row = team_summ.loc[team]
        t_i = team_to_idx[team]
        print(f"  {team:<28s}  {row['obs']:>7.2f}  {row['pred']:>7.2f}  "
              f"{row['diff']:>+7.2f}  {alpha_mean[t_i]:>8.4f}")

team_obs_mean = (df.groupby('team_name')['points_scored']
                 .mean().sort_values(ascending=False))
top10_teams    = team_obs_mean.head(10).index.tolist()
bottom10_teams = team_obs_mean.tail(10).index.tolist()

print("=== Alpha_team detail — Options 2 and 3 ===")
alpha_team_detail("2", samp_2, training_data_opt2)
alpha_team_detail("3", samp_3, training_data_opt3)
print()


# ---------------------------------------------------------------------------
# Sigma_obs per conference — Options 1, 2, 3
# ---------------------------------------------------------------------------
print("=== sigma_obs posterior mean per conference ===")
print(f"  {'Conference':<22s}  {'Obs log-var':>11s}  "
      f"{'Opt1 sig':>9s}  {'Opt1 sig²':>9s}  "
      f"{'Opt2 sig':>9s}  {'Opt2 sig²':>9s}  "
      f"{'Opt3 sig':>9s}  {'Opt3 sig²':>9s}")
print("  " + "-" * 95)
for i, c in enumerate(CONFERENCES):
    s1 = float(np.array(samp_1['sigma_obs'])[:, i].mean())
    s2 = float(np.array(samp_2['sigma_obs'])[:, i].mean())
    s3 = float(np.array(samp_3['sigma_obs'])[:, i].mean())
    print(f"  {c:<22s}  {obs_log_var[c]:>11.4f}  "
          f"{s1:>9.4f}  {s1**2:>9.4f}  "
          f"{s2:>9.4f}  {s2**2:>9.4f}  "
          f"{s3:>9.4f}  {s3**2:>9.4f}")
print()


# ---------------------------------------------------------------------------
# Final comparison table
# ---------------------------------------------------------------------------
print("=" * 74)
print("OPTION COMPARISON — 1 / 2 / 3")
print("(1 chain, 1000 warmup, 1000 samples — R-hat undefined for 1 chain)")
print("=" * 74)
print(f"  {'Metric':<42s}  {'Opt 1':>8s}  {'Opt 2':>8s}  {'Opt 3':>8s}")
print("  " + "-" * 64)
print(f"  {'Description':<42s}  {'No-team':>8s}  {'Zero-ft':>8s}  {'Ctr-ft':>8s}")
print(f"  {'Fit time (s)':<42s}  {t_1:>8.0f}  {t_2:>8.0f}  {t_3:>8.0f}")
print(f"  {'Divergences':<42s}  {ndiv_1:>8d}  {ndiv_2:>8d}  {ndiv_3:>8d}")
print(f"  {'Min ESS_bulk (scalar params)':<42s}  {ess_1:>8.0f}  {ess_2:>8.0f}  {ess_3:>8.0f}")
print(f"  {'sigma_attack mean (Opt2/3 only)':<42s}  {'N/A':>8s}  {sa2_mean:>8.4f}  {sa3_mean:>8.4f}")
print(f"  {'sigma_attack median':<42s}  {'N/A':>8s}  {sa2_med:>8.4f}  {sa3_med:>8.4f}")
print(f"  {'sigma_attack std':<42s}  {'N/A':>8s}  {sa2_std:>8.4f}  {sa3_std:>8.4f}")
print(f"  {'Log-var check: confs passed / 10':<42s}  {vpass_1:>8d}  {vpass_2:>8d}  {vpass_3:>8d}")
print(f"  {'Team mean check: passed / 131':<42s}  {tp_1:>8d}  {tp_2:>8d}  {tp_3:>8d}")
print(f"  {'Team mean check: worst gap (pts)':<42s}  {wt_1:>8.2f}  {wt_2:>8.2f}  {wt_3:>8.2f}")
print(f"  {'Conf mean check: passed / 10':<42s}  {cp_1:>8d}  {cp_2:>8d}  {cp_3:>8d}")
print(f"  {'Conf mean check: worst gap (pts)':<42s}  {wc_1:>8.2f}  {wc_2:>8.2f}  {wc_3:>8.2f}")
print("=" * 74)
print()
print("Cell 12 complete.")

Option 3 — centered feature stats (expect mean≈0 within team-season):
  Feature                          Global mean  Global std       Min       Max
  sp_rating                            -0.0000      0.0000   -0.0000    0.0000
  pregame_elo                           0.0000      0.2181   -1.2345    1.0109
  last3_win_pct                        -0.0000      0.7080   -2.2904    2.4179
  last3_points_scored_avg              -0.0000      0.6781   -2.2208    2.7587
  last3_points_allowed_avg              0.0000      0.6963   -2.8811    2.8800

Option 2 — zeroed feature check (expect all zeros):
  sp_rating                       sum=0.0000
  pregame_elo                     sum=0.0000
  last3_win_pct                   sum=0.0000

Option 3 — centered feature check (expect mean≈0):
  sp_rating                       mean=-0.000000  std=0.0000
  pregame_elo                     mean=0.000000  std=0.2181
  last3_win_pct                   mean=-0.000000  std=0.7080

Running diagnostic fits (1 chain,

In [13]:
# =============================================================================
# CELL 13 — Combined fix diagnostic
#
# Three changes from model_08 architecture:
#
#   Fix 1 — Informative prior on sigma_attack and sigma_defense.
#            HalfNormal(0.25) -> LogNormal(-1.5, 0.5)
#            Prior median: exp(-1.5) = 0.22. 90th pct: ~0.50.
#            Encodes domain knowledge: CFB has large persistent team
#            quality differences. Oregon vs Michigan State = 0.86 log-scale.
#
#   Fix 2 — Offensive-defensive archetype interaction.
#            b_matchup[off_archetype_idx[i], def_archetype_idx[j]]
#            4x4 matrix (16 parameters). Captures how offensive scheme
#            performs against defensive scheme beyond additive main effects.
#            off_archetype_idx[i] = team i offensive style (0-3)
#            def_archetype_idx[j] = opponent j defensive style (0-3)
#            Both arrays already in GameData from Cell 5.
#            Need: opponent defensive archetype per row.
#
#   Fix 3 — Log-Normal likelihood.
#            log(points + 1) ~ Normal(log_mu, sigma_obs[conf_idx])
#            sigma_obs ~ HalfNormal(0.3) x N_CONFERENCES
#
# Data pipeline unchanged. Feature set unchanged.
# One new data array: opp_def_archetype_idx (opponent's def archetype).
# Built from df using opponent column and team_to_idx mapping.
# =============================================================================

import numpy as np
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value
from jax import random
import arviz as az
import dataclasses
import time

# ---------------------------------------------------------------------------
# Data preparation — opponent defensive archetype index
#
# For each row (team i vs opponent j), we need j's defensive archetype.
# df has: team_name, opponent, def_archetype_idx (team i's own def archetype)
# We need: for each row, the def_archetype_idx of the opponent team
#          in that same game.
#
# Method: build a map from (game_id, team_name) -> def_archetype_idx,
# then look up (game_id, opponent) for each row.
# ---------------------------------------------------------------------------

# Build lookup: (game_id, team_name) -> def_archetype_idx
game_team_to_def_arch = (
    df.set_index(['game_id', 'team_name'])['def_archetype_idx']
    .to_dict()
)

# For each row, look up opponent's def_archetype_idx in the same game
opp_def_arch_vals = []
for _, row in df.iterrows():
    key = (row['game_id'], row['opponent'])
    opp_def_arch_vals.append(game_team_to_def_arch[key])

opp_def_archetype_idx = jnp.array(opp_def_arch_vals, dtype=jnp.int32)

# Verify
assert opp_def_archetype_idx.shape == (len(df),), \
    f"Shape mismatch: {opp_def_archetype_idx.shape}"
assert int(opp_def_archetype_idx.min()) >= 0, "opp_def_archetype_idx below 0"
assert int(opp_def_archetype_idx.max()) <= 3, "opp_def_archetype_idx above 3"

print("Opponent defensive archetype index built.")
print(f"  shape : {opp_def_archetype_idx.shape}")
print(f"  range : [{int(opp_def_archetype_idx.min())}, "
      f"{int(opp_def_archetype_idx.max())}]")
print(f"  dtype : {opp_def_archetype_idx.dtype}")
print()

# Value counts — confirm all 4 archetypes present
unique, counts = np.unique(np.array(opp_def_archetype_idx), return_counts=True)
print("  Opponent def archetype distribution:")
for u, c in zip(unique, counts):
    print(f"    archetype {u} : {c:,} rows ({c/len(df)*100:.1f}%)")
print()

# Also verify team i's own off_archetype_idx distribution for comparison
unique_o, counts_o = np.unique(
    np.array(training_data.off_archetype_idx), return_counts=True
)
print("  Team off archetype distribution:")
for u, c in zip(unique_o, counts_o):
    print(f"    archetype {u} : {c:,} rows ({c/len(df)*100:.1f}%)")
print()

# Add opp_def_archetype_idx to GameData
# GameData dataclass already has off_archetype_idx and def_archetype_idx.
# We extend training_data with the new field by rebuilding with replace.
# Note: GameData does not have opp_def_archetype_idx as a field yet —
# we pass it separately to the model function rather than adding to GameData,
# to avoid rewriting the dataclass definition from Cell 2.
# This keeps Cell 2 untouched per project rules.

# ---------------------------------------------------------------------------
# Shared arrays for posterior predictive checks
# ---------------------------------------------------------------------------
log_points = jnp.array(
    np.log1p(np.array(df['points_scored'].values, dtype=np.float32)),
    dtype=jnp.float32
)

conf_labels   = np.array(df['conference'].values)
log_points_np = np.array(log_points)
points_np     = np.array(df['points_scored'].values, dtype=np.float32)

obs_log_var = {}
for c_name in CONFERENCES:
    mask  = conf_labels == c_name
    y_log = log_points_np[mask]
    obs_log_var[c_name] = float(y_log.var())

team_idx_arr      = np.array(team_idx,              dtype=np.int32)
opp_idx_arr       = np.array(opp_idx,               dtype=np.int32)
conf_idx_arr      = np.array(conf_idx,              dtype=np.int32)
is_home_arr       = np.array(df['is_home'].values,  dtype=np.float32)
opp_def_arch_arr  = np.array(opp_def_archetype_idx, dtype=np.int32)
off_arch_arr      = np.array(training_data.off_archetype_idx, dtype=np.int32)

# ---------------------------------------------------------------------------
# Model definition — combined fix
# ---------------------------------------------------------------------------
def model_fix(data: GameData, N_teams: int,
              log_points: jnp.ndarray,
              opp_def_archetype_idx: jnp.ndarray):

    # ── League level ────────────────────────────────────────────────────────
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))

    # ── Variance — Log-Normal likelihood ────────────────────────────────────
    sigma_obs = numpyro.sample(
        "sigma_obs", dist.HalfNormal(0.3), sample_shape=(N_CONFERENCES,)
    )

    # ── Conference level ────────────────────────────────────────────────────
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(0.1))
    mu_conference    = numpyro.sample(
        "mu_conference", dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )

    # ── Team level — non-centered, Fix 1: LogNormal prior on sigma ──────────
    # LogNormal(-1.5, 0.5): median=0.22, 90th pct=0.50
    # Encodes: persistent team quality differences in CFB are large.
    # Oregon vs Michigan State = 0.86 log-scale gap over 3 seasons.
    sigma_attack   = numpyro.sample("sigma_attack",   dist.LogNormal(-1.5, 0.5))
    alpha_team_raw = numpyro.sample(
        "alpha_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    alpha_team = numpyro.deterministic("alpha_team", alpha_team_raw * sigma_attack)

    sigma_defense  = numpyro.sample("sigma_defense",  dist.LogNormal(-1.5, 0.5))
    delta_team_raw = numpyro.sample(
        "delta_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    delta_team = numpyro.deterministic("delta_team", delta_team_raw * sigma_defense)

    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team_raw   = numpyro.sample(
        "hfa_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    hfa_team = numpyro.deterministic("hfa_team", hfa_team_raw * sigma_hfa_team)

    # ── Prior seeds ─────────────────────────────────────────────────────────
    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 0.15))

    rec_weight_other   = numpyro.sample(
        "rec_weight_other", dist.Normal(0.0, 0.5),
        sample_shape=(N_CONFERENCES - 1,)
    )
    rec_weight_sunbelt = numpyro.sample(
        "rec_weight_sunbelt", dist.TruncatedNormal(0.0, 0.5, high=0.0)
    )
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])

    # ── Game-level coefficients ─────────────────────────────────────────────
    b_close_game_epa       = numpyro.sample("b_close_game_epa",       dist.Normal(0.0, 0.10))
    b_close_game_def_epa   = numpyro.sample("b_close_game_def_epa",   dist.Normal(0.0, 0.10))
    b_pregame_elo          = numpyro.sample("b_pregame_elo",          dist.Normal(0.0, 0.15))
    b_elo_sp_divergence    = numpyro.sample("b_elo_sp_divergence",    dist.Normal(0.0, 0.15))
    b_last3_win_pct        = numpyro.sample("b_last3_win_pct",        dist.Normal(0.0, 0.15))
    b_away_travel_distance = numpyro.sample("b_away_travel_distance", dist.Normal(0.0, 0.15))
    b_away_tz_delta        = numpyro.sample("b_away_tz_delta",        dist.Normal(0.0, 0.15))
    b_wind_chill           = numpyro.sample("b_wind_chill",           dist.Normal(0.0, 0.15))
    b_rush_rate_std_downs  = numpyro.sample("b_rush_rate_std_downs",  dist.Normal(0.0, 0.15))
    b_rush_rate_pass_downs = numpyro.sample("b_rush_rate_pass_downs", dist.Normal(0.0, 0.15))
    b_off_sack_rate_allowed= numpyro.sample("b_off_sack_rate_allowed",dist.Normal(0.0, 0.15))
    b_def_sack_rate        = numpyro.sample("b_def_sack_rate",        dist.Normal(0.0, 0.15))

    # ── Archetype main effects ──────────────────────────────────────────────
    b_off_archetype = numpyro.sample(
        "b_off_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_def_archetype = numpyro.sample(
        "b_def_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )

    # ── Fix 2: Offensive-defensive archetype interaction ────────────────────
    # b_matchup[i, j]: how offensive archetype i scores against def archetype j
    # relative to the additive main effects above.
    # 4x4 = 16 parameters. Prior Normal(0, 0.10) — tighter than main effects
    # because interaction terms should be smaller than main effects.
    b_matchup = numpyro.sample(
        "b_matchup", dist.Normal(0.0, 0.10), sample_shape=(4, 4)
    )

    # ── Conference-scoped features ──────────────────────────────────────────
    b_last3_off_epa_avg      = numpyro.sample("b_last3_off_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_def_epa_avg      = numpyro.sample("b_last3_def_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_pts_scored_avg   = numpyro.sample("b_last3_pts_scored_avg",   dist.Normal(0.0, 0.15))
    b_last3_pts_allowed_avg  = numpyro.sample("b_last3_pts_allowed_avg",  dist.Normal(0.0, 0.15))
    b_days_since_last_game   = numpyro.sample("b_days_since_last_game",   dist.Normal(0.0, 0.15))
    b_close_play_count_delta = numpyro.sample("b_close_play_count_delta", dist.Normal(0.0, 0.15))
    b_away_elevation_delta   = numpyro.sample("b_away_elevation_delta",   dist.Normal(0.0, 0.15))

    # ── Linear predictor ────────────────────────────────────────────────────
    log_mu = (
        mu_league
        + mu_conference[data.conf_idx]
        + alpha_team[data.team_idx]
        - delta_team[data.opp_idx]
        + (hfa_league + hfa_team[data.team_idx]) * data.is_home
        + sp_weight              * data.sp_rating
        + rec_weight[data.conf_idx] * data.recruiting_3yr_avg
        + b_close_game_epa       * data.close_game_epa
        + b_close_game_def_epa   * data.close_game_def_epa
        + b_pregame_elo          * data.pregame_elo
        + b_elo_sp_divergence    * data.elo_sp_divergence
        + b_last3_win_pct        * data.last3_win_pct
        + b_away_travel_distance * data.away_travel_distance
        + b_away_tz_delta        * data.away_tz_delta
        + b_wind_chill           * data.wind_chill
        + b_rush_rate_std_downs  * data.rush_rate_std_downs
        + b_rush_rate_pass_downs * data.rush_rate_pass_downs
        + b_off_sack_rate_allowed * data.off_sack_rate_allowed
        + b_def_sack_rate         * data.def_sack_rate
        # Archetype main effects
        + b_off_archetype[data.off_archetype_idx]
        + b_def_archetype[data.def_archetype_idx]
        # Fix 2: matchup interaction — off archetype i vs opp def archetype j
        + b_matchup[data.off_archetype_idx, opp_def_archetype_idx]
        # Conference-scoped features
        + b_last3_off_epa_avg     * data.mask_last3_off_epa    * data.last3_off_epa_avg
        + b_last3_def_epa_avg     * data.mask_last3_def_epa    * data.last3_def_epa_avg
        + b_last3_pts_scored_avg  * data.mask_last3_pts_scored * data.last3_pts_scored_avg
        + b_last3_pts_allowed_avg * data.mask_last3_pts_allowed * data.last3_pts_allowed_avg
        + b_days_since_last_game  * data.mask_days_since        * data.days_since_last_game
        + b_close_play_count_delta * data.mask_close_play_count * data.close_play_count_delta
        + b_away_elevation_delta  * data.mask_elevation         * data.away_elevation_delta
    )

    # ── Likelihood — Fix 3: Log-Normal ──────────────────────────────────────
    numpyro.sample(
        "obs",
        dist.Normal(log_mu, sigma_obs[data.conf_idx]),
        obs=log_points
    )


# ---------------------------------------------------------------------------
# Verify model prints expected prior spec before fitting
# ---------------------------------------------------------------------------
print("model_fix() defined.")
print("  sigma_attack    : LogNormal(-1.5, 0.5)  "
      f"[median={float(jnp.exp(jnp.array(-1.5))):.3f}, "
      f"p90={float(jnp.exp(jnp.array(-1.5 + 1.28*0.5))):.3f}]")
print("  sigma_defense   : LogNormal(-1.5, 0.5)")
print("  b_matchup       : Normal(0.0, 0.10) x (4, 4)  [16 parameters]")
print("  sigma_obs       : HalfNormal(0.3) x N_CONFERENCES")
print("  likelihood      : Normal(log_mu, sigma_obs[conf_idx])")
print("  obs target      : log(points + 1)")
print()


# ---------------------------------------------------------------------------
# Init values
# ---------------------------------------------------------------------------
init_fix = {
    "mu_league"          : jnp.array(3.3),
    "hfa_league"         : jnp.array(0.1),
    "sigma_obs"          : jnp.ones(N_CONFERENCES) * 0.3,
    "sigma_conference"   : jnp.array(0.05),
    "mu_conference"      : jnp.zeros(N_CONFERENCES),
    # Fix 1: start sigma_attack at prior median, not near zero
    "sigma_attack"       : jnp.array(0.22),
    "alpha_team_raw"     : jnp.zeros(N_teams),
    "sigma_defense"      : jnp.array(0.22),
    "delta_team_raw"     : jnp.zeros(N_teams),
    "sigma_hfa_team"     : jnp.array(0.05),
    "hfa_team_raw"       : jnp.zeros(N_teams),
    "sp_weight"          : jnp.array(0.0),
    "rec_weight_other"   : jnp.zeros(N_CONFERENCES - 1),
    "rec_weight_sunbelt" : jnp.array(-0.1),
    "b_close_game_epa"        : jnp.array(0.0),
    "b_close_game_def_epa"    : jnp.array(0.0),
    "b_pregame_elo"           : jnp.array(0.0),
    "b_elo_sp_divergence"     : jnp.array(0.0),
    "b_last3_win_pct"         : jnp.array(0.0),
    "b_away_travel_distance"  : jnp.array(0.0),
    "b_away_tz_delta"         : jnp.array(0.0),
    "b_wind_chill"            : jnp.array(0.0),
    "b_rush_rate_std_downs"   : jnp.array(0.0),
    "b_rush_rate_pass_downs"  : jnp.array(0.0),
    "b_off_sack_rate_allowed" : jnp.array(0.0),
    "b_def_sack_rate"         : jnp.array(0.0),
    "b_off_archetype"         : jnp.zeros(4),
    "b_def_archetype"         : jnp.zeros(4),
    "b_matchup"               : jnp.zeros((4, 4)),
    "b_last3_off_epa_avg"     : jnp.array(0.0),
    "b_last3_def_epa_avg"     : jnp.array(0.0),
    "b_last3_pts_scored_avg"  : jnp.array(0.0),
    "b_last3_pts_allowed_avg" : jnp.array(0.0),
    "b_days_since_last_game"  : jnp.array(0.0),
    "b_close_play_count_delta": jnp.array(0.0),
    "b_away_elevation_delta"  : jnp.array(0.0),
}


# ---------------------------------------------------------------------------
# Fit — 1 chain, 1000 warmup, 1000 samples
# ---------------------------------------------------------------------------
print("Fitting model_fix (1 chain, 1000 warmup, 1000 samples)...",
      flush=True)

nuts_fix = NUTS(
    model_fix,
    target_accept_prob=0.9,
    init_strategy=init_to_value(values=init_fix),
)
mcmc_fix = MCMC(
    nuts_fix,
    num_warmup=1000,
    num_samples=1000,
    num_chains=1,
    progress_bar=False,
)

numpyro.enable_validation(False)
try:
    t0 = time.time()
    mcmc_fix.run(
        random.PRNGKey(0),
        data=training_data,
        N_teams=N_teams,
        log_points=log_points,
        opp_def_archetype_idx=opp_def_archetype_idx,
    )
    t_fix = time.time() - t0
finally:
    numpyro.enable_validation(True)

extra_fix = mcmc_fix.get_extra_fields()
ndiv_fix  = int(extra_fix['diverging'].sum()) if 'diverging' in extra_fix else 0
samp_fix  = mcmc_fix.get_samples()
idata_fix = az.from_numpyro(mcmc_fix)

print(f"done ({t_fix:.0f}s)  divergences={ndiv_fix}")
print()


# ---------------------------------------------------------------------------
# Convergence — ESS_bulk via az.ess
# ---------------------------------------------------------------------------
ess_vars = ["mu_league", "hfa_league", "sigma_conference",
            "sigma_attack", "sigma_defense", "sigma_hfa_team",
            "sp_weight", "b_close_game_epa"]
ess_fix_vals = az.ess(idata_fix, var_names=ess_vars)
min_ess_fix  = min(float(ess_fix_vals[v].values.min())
                   for v in ess_vars if v in ess_fix_vals)


# ---------------------------------------------------------------------------
# sigma_attack and sigma_defense posterior summary
# ---------------------------------------------------------------------------
sa_fix = np.array(samp_fix['sigma_attack'])
sd_fix = np.array(samp_fix['sigma_defense'])

print("=== sigma_attack posterior (Fix 1 — LogNormal prior) ===")
print(f"  mean   : {sa_fix.mean():.4f}  (was 0.025 in all previous models)")
print(f"  median : {np.percentile(sa_fix, 50):.4f}")
print(f"  std    : {sa_fix.std():.4f}")
print(f"  5th    : {np.percentile(sa_fix, 5):.4f}")
print(f"  95th   : {np.percentile(sa_fix, 95):.4f}")
print(f"  Pct<0.05: {float((sa_fix < 0.05).mean())*100:.1f}%")
print()
print("=== sigma_defense posterior ===")
print(f"  mean   : {sd_fix.mean():.4f}")
print(f"  median : {np.percentile(sd_fix, 50):.4f}")
print(f"  5th    : {np.percentile(sd_fix, 5):.4f}")
print(f"  95th   : {np.percentile(sd_fix, 95):.4f}")
print()


# ---------------------------------------------------------------------------
# b_matchup posterior means — 4x4 table
# ---------------------------------------------------------------------------
b_matchup_mean = np.array(samp_fix['b_matchup']).mean(axis=0)  # (4, 4)
print("=== b_matchup posterior means (off_arch x opp_def_arch) ===")
print(f"  {'':12s}  {'Def 0':>8s}  {'Def 1':>8s}  {'Def 2':>8s}  {'Def 3':>8s}")
print("  " + "-" * 48)
for i in range(4):
    row_str = "  ".join(f"{b_matchup_mean[i, j]:>8.4f}" for j in range(4))
    print(f"  Off {i:<8d}  {row_str}")
print()


# ---------------------------------------------------------------------------
# Log-variance check — 100 draws
# ---------------------------------------------------------------------------
def logvar_check_fix(samp_):
    mu_l  = np.array(samp_['mu_league'])
    hfa_l = np.array(samp_['hfa_league'])
    mu_c  = np.array(samp_['mu_conference'])
    alp   = np.array(samp_['alpha_team'])
    dlt   = np.array(samp_['delta_team'])
    hfa   = np.array(samp_['hfa_team'])
    sig   = np.array(samp_['sigma_obs'])
    bm    = np.array(samp_['b_matchup'])   # (S, 4, 4)
    rng   = np.random.default_rng(0)
    draws = rng.choice(len(mu_l), size=100, replace=False)
    conf_draws = {c: [] for c in CONFERENCES}
    for d in draws:
        log_mu = (
            mu_l[d]
            + mu_c[d, conf_idx_arr]
            + alp[d, team_idx_arr]
            - dlt[d, opp_idx_arr]
            + hfa_l[d] * is_home_arr
            + hfa[d, team_idx_arr] * is_home_arr
            + bm[d, off_arch_arr, opp_def_arch_arr]
        )
        sig_o  = sig[d, conf_idx_arr]
        y_log  = rng.normal(loc=log_mu, scale=sig_o)
        for c in CONFERENCES:
            mask = conf_labels == c
            conf_draws[c].append(y_log[mask].var())
    n_pass = 0
    for c in CONFERENCES:
        dc = np.array(conf_draws[c])
        lo, hi = np.percentile(dc, 5), np.percentile(dc, 95)
        if lo <= obs_log_var[c] <= hi:
            n_pass += 1
    return n_pass

print("Running log-variance check...", flush=True)
vpass_fix = logvar_check_fix(samp_fix)
print(f"done.  Conferences passed: {vpass_fix} / 10")
print()


# ---------------------------------------------------------------------------
# Team-level and conference-level mean checks
# ---------------------------------------------------------------------------
mu_l_fix  = np.array(samp_fix['mu_league']).mean()
hfa_l_fix = np.array(samp_fix['hfa_league']).mean()
mu_c_fix  = np.array(samp_fix['mu_conference']).mean(axis=0)
alp_fix   = np.array(samp_fix['alpha_team']).mean(axis=0)
dlt_fix   = np.array(samp_fix['delta_team']).mean(axis=0)
hfa_fix   = np.array(samp_fix['hfa_team']).mean(axis=0)
sig_fix   = np.array(samp_fix['sigma_obs']).mean(axis=0)
bm_fix    = np.array(samp_fix['b_matchup']).mean(axis=0)

log_mu_fix = (
    mu_l_fix
    + mu_c_fix[conf_idx_arr]
    + alp_fix[team_idx_arr]
    - dlt_fix[opp_idx_arr]
    + hfa_l_fix * is_home_arr
    + hfa_fix[team_idx_arr] * is_home_arr
    + bm_fix[off_arch_arr, opp_def_arch_arr]
)
sig_o_fix  = sig_fix[conf_idx_arr]
mu_pred_fix = np.exp(log_mu_fix + 0.5 * sig_o_fix**2) - 1.0

tmp_fix = df[['team_name', 'conference', 'points_scored']].copy()
tmp_fix['mu_pred'] = mu_pred_fix

team_summ_fix = (tmp_fix.groupby('team_name')
                 .agg(obs=('points_scored', 'mean'),
                      pred=('mu_pred', 'mean'))
                 .assign(diff=lambda x: x['pred'] - x['obs']))
conf_summ_fix = (tmp_fix.groupby('conference')
                 .agg(obs=('points_scored', 'mean'),
                      pred=('mu_pred', 'mean'))
                 .assign(diff=lambda x: x['pred'] - x['obs']))

tp_fix = int((team_summ_fix['diff'].abs() <= 2.0).sum())
cp_fix = int((conf_summ_fix['diff'].abs() <= 3.0).sum())
wt_fix = float(team_summ_fix['diff'].abs().max())
wc_fix = float(conf_summ_fix['diff'].abs().max())

# Top 10 and bottom 10 team detail
team_obs_mean  = (df.groupby('team_name')['points_scored']
                  .mean().sort_values(ascending=False))
top10_teams    = team_obs_mean.head(10).index.tolist()
bottom10_teams = team_obs_mean.tail(10).index.tolist()

alpha_fix_mean = alp_fix  # already posterior mean

print("=== Team-level mean check — top 10 and bottom 10 ===")
print(f"  {'Team':<28s}  {'Obs':>7s}  {'Pred':>7s}  {'Diff':>7s}  {'alpha':>8s}")
print("  " + "-" * 62)
for team in top10_teams:
    row = team_summ_fix.loc[team]
    t_i = team_to_idx[team]
    print(f"  {team:<28s}  {row['obs']:>7.2f}  {row['pred']:>7.2f}  "
          f"{row['diff']:>+7.2f}  {alpha_fix_mean[t_i]:>8.4f}")
print("  ...")
for team in bottom10_teams:
    row = team_summ_fix.loc[team]
    t_i = team_to_idx[team]
    print(f"  {team:<28s}  {row['obs']:>7.2f}  {row['pred']:>7.2f}  "
          f"{row['diff']:>+7.2f}  {alpha_fix_mean[t_i]:>8.4f}")
print()

print("=== Conference-level mean check ===")
print(f"  {'Conference':<22s}  {'Obs':>7s}  {'Pred':>7s}  {'Diff':>7s}  Status")
print("  " + "-" * 55)
for c_name, row in conf_summ_fix.iterrows():
    status = "PASS" if abs(row['diff']) <= 3.0 else "FLAG"
    print(f"  {c_name:<22s}  {row['obs']:>7.2f}  {row['pred']:>7.2f}  "
          f"{row['diff']:>+7.2f}  {status}")
print()

# sigma_obs per conference
print("=== sigma_obs posterior mean per conference ===")
print(f"  {'Conference':<22s}  {'Obs log-var':>11s}  "
      f"{'sigma_obs':>10s}  {'sigma²_obs':>10s}  {'Gap':>8s}")
print("  " + "-" * 68)
sig_obs_draws = np.array(samp_fix['sigma_obs'])
for i, c in enumerate(CONFERENCES):
    s  = float(sig_obs_draws[:, i].mean())
    s2 = s ** 2
    gap = obs_log_var[c] - s2
    print(f"  {c:<22s}  {obs_log_var[c]:>11.4f}  "
          f"{s:>10.4f}  {s2:>10.4f}  {gap:>+8.4f}")
print()


# ---------------------------------------------------------------------------
# Summary table
# ---------------------------------------------------------------------------
print("=" * 60)
print("COMBINED FIX DIAGNOSTIC SUMMARY")
print("(1 chain, 1000 warmup, 1000 samples)")
print("=" * 60)
print(f"  Fit time (s)                    : {t_fix:.0f}")
print(f"  Divergences                     : {ndiv_fix}")
print(f"  Min ESS_bulk (scalar params)    : {min_ess_fix:.0f}")
print(f"  sigma_attack mean               : {sa_fix.mean():.4f}  "
      f"(target: 0.15-0.30)")
print(f"  sigma_defense mean              : {sd_fix.mean():.4f}")
print(f"  Log-var check: passed / 10      : {vpass_fix}")
print(f"  Team mean check: passed / 131   : {tp_fix}  "
      f"(prev best: 37)")
print(f"  Team mean check: worst gap      : {wt_fix:.2f} pts  "
      f"(prev best: 14.51)")
print(f"  Conf mean check: passed / 10    : {cp_fix}  "
      f"(prev best: 7)")
print(f"  Conf mean check: worst gap      : {wc_fix:.2f} pts  "
      f"(prev best: 4.85)")
print("=" * 60)
print()
print("Cell 13 complete.")

Opponent defensive archetype index built.
  shape : (3214,)
  range : [0, 3]
  dtype : int32

  Opponent def archetype distribution:
    archetype 0 : 691 rows (21.5%)
    archetype 1 : 505 rows (15.7%)
    archetype 2 : 909 rows (28.3%)
    archetype 3 : 1,109 rows (34.5%)

  Team off archetype distribution:
    archetype 0 : 1,282 rows (39.9%)
    archetype 1 : 1,044 rows (32.5%)
    archetype 2 : 805 rows (25.0%)
    archetype 3 : 83 rows (2.6%)

model_fix() defined.
  sigma_attack    : LogNormal(-1.5, 0.5)  [median=0.223, p90=0.423]
  sigma_defense   : LogNormal(-1.5, 0.5)
  b_matchup       : Normal(0.0, 0.10) x (4, 4)  [16 parameters]
  sigma_obs       : HalfNormal(0.3) x N_CONFERENCES
  likelihood      : Normal(log_mu, sigma_obs[conf_idx])
  obs target      : log(points + 1)

Fitting model_fix (1 chain, 1000 warmup, 1000 samples)...
done (95s)  divergences=0

=== sigma_attack posterior (Fix 1 — LogNormal prior) ===
  mean   : 0.0520  (was 0.025 in all previous models)
  median : 

In [14]:
# =============================================================================
# CELL 14 — Model E vs Model F diagnostic comparison
#
# Model E — Features only, no team intercepts.
#   alpha_team, delta_team, sigma_attack, sigma_defense: REMOVED.
#   Features carry all team quality signal.
#   Log-Normal likelihood. sigma_obs per conference.
#   Matchup interaction retained.
#
# Model F — Team-season intercepts.
#   alpha_team[team] replaced with alpha_team_season[team_season_idx].
#   393 intercepts (131 teams x 3 seasons). No static pooling across seasons.
#   Same features as Model E plus team-season intercepts.
#   Log-Normal likelihood. sigma_obs per conference.
#   Matchup interaction retained.
#
# Diagnostic fit: 1 chain, 1000 warmup, 1000 samples, PRNGKey(0).
# =============================================================================

import numpy as np
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value
from jax import random
import arviz as az
import time

# ---------------------------------------------------------------------------
# Data prep — team-season index for Model F
# ---------------------------------------------------------------------------
seasons       = sorted(df['season'].astype(int).unique())   # [2022, 2023, 2024]
season_to_idx = {s: i for i, s in enumerate(seasons)}

# All (team, season) pairs present in training data
team_season_pairs = sorted(set(
    zip(df['team_name'], df['season'].astype(int))
))
ts_to_idx      = {(t, s): i for i, (t, s) in enumerate(team_season_pairs)}
N_team_seasons = len(ts_to_idx)

print(f"Team-season index built.")
print(f"  Seasons          : {seasons}")
print(f"  N_team_seasons   : {N_team_seasons}  (expect 131 x 3 = 393)")
print()

# team_season_idx: for each row, index of (team_name, season)
team_season_idx = jnp.array(
    [ts_to_idx[(row.team_name, int(row.season))]
     for _, row in df.iterrows()],
    dtype=jnp.int32
)

# opp_season_idx: for each row, index of (opponent, season)
opp_season_idx = jnp.array(
    [ts_to_idx[(row.opponent, int(row.season))]
     for _, row in df.iterrows()],
    dtype=jnp.int32
)

assert team_season_idx.shape == (len(df),)
assert opp_season_idx.shape  == (len(df),)
assert int(team_season_idx.min()) == 0
assert int(team_season_idx.max()) == N_team_seasons - 1
assert int(opp_season_idx.min())  == 0
assert int(opp_season_idx.max())  == N_team_seasons - 1

print(f"  team_season_idx  : shape={team_season_idx.shape}  "
      f"range=[{int(team_season_idx.min())}, {int(team_season_idx.max())}]")
print(f"  opp_season_idx   : shape={opp_season_idx.shape}  "
      f"range=[{int(opp_season_idx.min())}, {int(opp_season_idx.max())}]")
print()

# ---------------------------------------------------------------------------
# Shared data arrays
# ---------------------------------------------------------------------------
log_points = jnp.array(
    np.log1p(np.array(df['points_scored'].values, dtype=np.float32)),
    dtype=jnp.float32
)

conf_labels   = np.array(df['conference'].values)
log_points_np = np.array(log_points)
points_np     = np.array(df['points_scored'].values, dtype=np.float32)

obs_log_var = {}
for c_name in CONFERENCES:
    mask = conf_labels == c_name
    obs_log_var[c_name] = float(log_points_np[mask].var())

team_idx_arr     = np.array(team_idx,              dtype=np.int32)
opp_idx_arr      = np.array(opp_idx,               dtype=np.int32)
conf_idx_arr     = np.array(conf_idx,              dtype=np.int32)
is_home_arr      = np.array(df['is_home'].values,  dtype=np.float32)
off_arch_arr     = np.array(training_data.off_archetype_idx, dtype=np.int32)
opp_def_arch_arr = np.array(opp_def_archetype_idx, dtype=np.int32)
ts_idx_arr       = np.array(team_season_idx,       dtype=np.int32)
os_idx_arr       = np.array(opp_season_idx,        dtype=np.int32)

# ---------------------------------------------------------------------------
# Shared linear predictor block (features only — no team intercepts)
# Used by both models. Team intercept terms added on top in Model F.
# ---------------------------------------------------------------------------
def _feature_log_mu(data, mu_league, hfa_league, mu_conference,
                    hfa_team, sp_weight, rec_weight,
                    b_close_game_epa, b_close_game_def_epa,
                    b_pregame_elo, b_elo_sp_divergence, b_last3_win_pct,
                    b_away_travel_distance, b_away_tz_delta, b_wind_chill,
                    b_rush_rate_std_downs, b_rush_rate_pass_downs,
                    b_off_sack_rate_allowed, b_def_sack_rate,
                    b_off_archetype, b_def_archetype, b_matchup,
                    opp_def_archetype_idx,
                    b_last3_off_epa_avg, b_last3_def_epa_avg,
                    b_last3_pts_scored_avg, b_last3_pts_allowed_avg,
                    b_days_since_last_game, b_close_play_count_delta,
                    b_away_elevation_delta):
    return (
        mu_league
        + mu_conference[data.conf_idx]
        + (hfa_league + hfa_team[data.team_idx]) * data.is_home
        + sp_weight              * data.sp_rating
        + rec_weight[data.conf_idx] * data.recruiting_3yr_avg
        + b_close_game_epa       * data.close_game_epa
        + b_close_game_def_epa   * data.close_game_def_epa
        + b_pregame_elo          * data.pregame_elo
        + b_elo_sp_divergence    * data.elo_sp_divergence
        + b_last3_win_pct        * data.last3_win_pct
        + b_away_travel_distance * data.away_travel_distance
        + b_away_tz_delta        * data.away_tz_delta
        + b_wind_chill           * data.wind_chill
        + b_rush_rate_std_downs  * data.rush_rate_std_downs
        + b_rush_rate_pass_downs * data.rush_rate_pass_downs
        + b_off_sack_rate_allowed * data.off_sack_rate_allowed
        + b_def_sack_rate         * data.def_sack_rate
        + b_off_archetype[data.off_archetype_idx]
        + b_def_archetype[data.def_archetype_idx]
        + b_matchup[data.off_archetype_idx, opp_def_archetype_idx]
        + b_last3_off_epa_avg     * data.mask_last3_off_epa    * data.last3_off_epa_avg
        + b_last3_def_epa_avg     * data.mask_last3_def_epa    * data.last3_def_epa_avg
        + b_last3_pts_scored_avg  * data.mask_last3_pts_scored * data.last3_pts_scored_avg
        + b_last3_pts_allowed_avg * data.mask_last3_pts_allowed * data.last3_pts_allowed_avg
        + b_days_since_last_game  * data.mask_days_since        * data.days_since_last_game
        + b_close_play_count_delta * data.mask_close_play_count * data.close_play_count_delta
        + b_away_elevation_delta  * data.mask_elevation         * data.away_elevation_delta
    )


def _sample_shared_params(N_teams):
    """Sample all parameters shared between Model E and Model F."""
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))

    sigma_obs = numpyro.sample(
        "sigma_obs", dist.HalfNormal(0.3), sample_shape=(N_CONFERENCES,)
    )
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(0.1))
    mu_conference    = numpyro.sample(
        "mu_conference", dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )
    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team_raw   = numpyro.sample(
        "hfa_team_raw", dist.Normal(0.0, 1.0), sample_shape=(N_teams,)
    )
    hfa_team = numpyro.deterministic("hfa_team", hfa_team_raw * sigma_hfa_team)

    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 0.15))
    rec_weight_other   = numpyro.sample(
        "rec_weight_other", dist.Normal(0.0, 0.5),
        sample_shape=(N_CONFERENCES - 1,)
    )
    rec_weight_sunbelt = numpyro.sample(
        "rec_weight_sunbelt", dist.TruncatedNormal(0.0, 0.5, high=0.0)
    )
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])

    b_close_game_epa       = numpyro.sample("b_close_game_epa",       dist.Normal(0.0, 0.10))
    b_close_game_def_epa   = numpyro.sample("b_close_game_def_epa",   dist.Normal(0.0, 0.10))
    b_pregame_elo          = numpyro.sample("b_pregame_elo",          dist.Normal(0.0, 0.15))
    b_elo_sp_divergence    = numpyro.sample("b_elo_sp_divergence",    dist.Normal(0.0, 0.15))
    b_last3_win_pct        = numpyro.sample("b_last3_win_pct",        dist.Normal(0.0, 0.15))
    b_away_travel_distance = numpyro.sample("b_away_travel_distance", dist.Normal(0.0, 0.15))
    b_away_tz_delta        = numpyro.sample("b_away_tz_delta",        dist.Normal(0.0, 0.15))
    b_wind_chill           = numpyro.sample("b_wind_chill",           dist.Normal(0.0, 0.15))
    b_rush_rate_std_downs  = numpyro.sample("b_rush_rate_std_downs",  dist.Normal(0.0, 0.15))
    b_rush_rate_pass_downs = numpyro.sample("b_rush_rate_pass_downs", dist.Normal(0.0, 0.15))
    b_off_sack_rate_allowed= numpyro.sample("b_off_sack_rate_allowed",dist.Normal(0.0, 0.15))
    b_def_sack_rate        = numpyro.sample("b_def_sack_rate",        dist.Normal(0.0, 0.15))

    b_off_archetype = numpyro.sample(
        "b_off_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_def_archetype = numpyro.sample(
        "b_def_archetype", dist.Normal(0.0, 0.15), sample_shape=(4,)
    )
    b_matchup = numpyro.sample(
        "b_matchup", dist.Normal(0.0, 0.10), sample_shape=(4, 4)
    )

    b_last3_off_epa_avg      = numpyro.sample("b_last3_off_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_def_epa_avg      = numpyro.sample("b_last3_def_epa_avg",      dist.Normal(0.0, 0.15))
    b_last3_pts_scored_avg   = numpyro.sample("b_last3_pts_scored_avg",   dist.Normal(0.0, 0.15))
    b_last3_pts_allowed_avg  = numpyro.sample("b_last3_pts_allowed_avg",  dist.Normal(0.0, 0.15))
    b_days_since_last_game   = numpyro.sample("b_days_since_last_game",   dist.Normal(0.0, 0.15))
    b_close_play_count_delta = numpyro.sample("b_close_play_count_delta", dist.Normal(0.0, 0.15))
    b_away_elevation_delta   = numpyro.sample("b_away_elevation_delta",   dist.Normal(0.0, 0.15))

    return (mu_league, hfa_league, sigma_obs, mu_conference, hfa_team,
            sp_weight, rec_weight,
            b_close_game_epa, b_close_game_def_epa, b_pregame_elo,
            b_elo_sp_divergence, b_last3_win_pct, b_away_travel_distance,
            b_away_tz_delta, b_wind_chill, b_rush_rate_std_downs,
            b_rush_rate_pass_downs, b_off_sack_rate_allowed, b_def_sack_rate,
            b_off_archetype, b_def_archetype, b_matchup,
            b_last3_off_epa_avg, b_last3_def_epa_avg,
            b_last3_pts_scored_avg, b_last3_pts_allowed_avg,
            b_days_since_last_game, b_close_play_count_delta,
            b_away_elevation_delta)


# ---------------------------------------------------------------------------
# MODEL E — Features only, no team intercepts
# ---------------------------------------------------------------------------
def model_e(data: GameData, N_teams: int,
            log_points: jnp.ndarray,
            opp_def_archetype_idx: jnp.ndarray):

    (mu_league, hfa_league, sigma_obs, mu_conference, hfa_team,
     sp_weight, rec_weight,
     b_close_game_epa, b_close_game_def_epa, b_pregame_elo,
     b_elo_sp_divergence, b_last3_win_pct, b_away_travel_distance,
     b_away_tz_delta, b_wind_chill, b_rush_rate_std_downs,
     b_rush_rate_pass_downs, b_off_sack_rate_allowed, b_def_sack_rate,
     b_off_archetype, b_def_archetype, b_matchup,
     b_last3_off_epa_avg, b_last3_def_epa_avg,
     b_last3_pts_scored_avg, b_last3_pts_allowed_avg,
     b_days_since_last_game, b_close_play_count_delta,
     b_away_elevation_delta) = _sample_shared_params(N_teams)

    log_mu = _feature_log_mu(
        data, mu_league, hfa_league, mu_conference, hfa_team,
        sp_weight, rec_weight,
        b_close_game_epa, b_close_game_def_epa, b_pregame_elo,
        b_elo_sp_divergence, b_last3_win_pct, b_away_travel_distance,
        b_away_tz_delta, b_wind_chill, b_rush_rate_std_downs,
        b_rush_rate_pass_downs, b_off_sack_rate_allowed, b_def_sack_rate,
        b_off_archetype, b_def_archetype, b_matchup, opp_def_archetype_idx,
        b_last3_off_epa_avg, b_last3_def_epa_avg,
        b_last3_pts_scored_avg, b_last3_pts_allowed_avg,
        b_days_since_last_game, b_close_play_count_delta,
        b_away_elevation_delta
    )

    numpyro.sample("obs",
                   dist.Normal(log_mu, sigma_obs[data.conf_idx]),
                   obs=log_points)


# ---------------------------------------------------------------------------
# MODEL F — Team-season intercepts
# ---------------------------------------------------------------------------
def model_f(data: GameData, N_teams: int,
            log_points: jnp.ndarray,
            opp_def_archetype_idx: jnp.ndarray,
            team_season_idx: jnp.ndarray,
            opp_season_idx: jnp.ndarray,
            N_team_seasons: int):

    (mu_league, hfa_league, sigma_obs, mu_conference, hfa_team,
     sp_weight, rec_weight,
     b_close_game_epa, b_close_game_def_epa, b_pregame_elo,
     b_elo_sp_divergence, b_last3_win_pct, b_away_travel_distance,
     b_away_tz_delta, b_wind_chill, b_rush_rate_std_downs,
     b_rush_rate_pass_downs, b_off_sack_rate_allowed, b_def_sack_rate,
     b_off_archetype, b_def_archetype, b_matchup,
     b_last3_off_epa_avg, b_last3_def_epa_avg,
     b_last3_pts_scored_avg, b_last3_pts_allowed_avg,
     b_days_since_last_game, b_close_play_count_delta,
     b_away_elevation_delta) = _sample_shared_params(N_teams)

    # Team-season intercepts
    sigma_attack      = numpyro.sample("sigma_attack",   dist.HalfNormal(0.25))
    alpha_ts_raw      = numpyro.sample(
        "alpha_ts_raw", dist.Normal(0.0, 1.0),
        sample_shape=(N_team_seasons,)
    )
    alpha_ts = numpyro.deterministic("alpha_ts", alpha_ts_raw * sigma_attack)

    sigma_defense     = numpyro.sample("sigma_defense",  dist.HalfNormal(0.25))
    delta_ts_raw      = numpyro.sample(
        "delta_ts_raw", dist.Normal(0.0, 1.0),
        sample_shape=(N_team_seasons,)
    )
    delta_ts = numpyro.deterministic("delta_ts", delta_ts_raw * sigma_defense)

    log_mu = (
        _feature_log_mu(
            data, mu_league, hfa_league, mu_conference, hfa_team,
            sp_weight, rec_weight,
            b_close_game_epa, b_close_game_def_epa, b_pregame_elo,
            b_elo_sp_divergence, b_last3_win_pct, b_away_travel_distance,
            b_away_tz_delta, b_wind_chill, b_rush_rate_std_downs,
            b_rush_rate_pass_downs, b_off_sack_rate_allowed, b_def_sack_rate,
            b_off_archetype, b_def_archetype, b_matchup, opp_def_archetype_idx,
            b_last3_off_epa_avg, b_last3_def_epa_avg,
            b_last3_pts_scored_avg, b_last3_pts_allowed_avg,
            b_days_since_last_game, b_close_play_count_delta,
            b_away_elevation_delta
        )
        + alpha_ts[team_season_idx]
        - delta_ts[opp_season_idx]
    )

    numpyro.sample("obs",
                   dist.Normal(log_mu, sigma_obs[data.conf_idx]),
                   obs=log_points)


# ---------------------------------------------------------------------------
# Init values
# ---------------------------------------------------------------------------
base_init = {
    "mu_league"          : jnp.array(3.3),
    "hfa_league"         : jnp.array(0.1),
    "sigma_obs"          : jnp.ones(N_CONFERENCES) * 0.3,
    "sigma_conference"   : jnp.array(0.05),
    "mu_conference"      : jnp.zeros(N_CONFERENCES),
    "sigma_hfa_team"     : jnp.array(0.05),
    "hfa_team_raw"       : jnp.zeros(N_teams),
    "sp_weight"          : jnp.array(0.0),
    "rec_weight_other"   : jnp.zeros(N_CONFERENCES - 1),
    "rec_weight_sunbelt" : jnp.array(-0.1),
    "b_close_game_epa"        : jnp.array(0.0),
    "b_close_game_def_epa"    : jnp.array(0.0),
    "b_pregame_elo"           : jnp.array(0.0),
    "b_elo_sp_divergence"     : jnp.array(0.0),
    "b_last3_win_pct"         : jnp.array(0.0),
    "b_away_travel_distance"  : jnp.array(0.0),
    "b_away_tz_delta"         : jnp.array(0.0),
    "b_wind_chill"            : jnp.array(0.0),
    "b_rush_rate_std_downs"   : jnp.array(0.0),
    "b_rush_rate_pass_downs"  : jnp.array(0.0),
    "b_off_sack_rate_allowed" : jnp.array(0.0),
    "b_def_sack_rate"         : jnp.array(0.0),
    "b_off_archetype"         : jnp.zeros(4),
    "b_def_archetype"         : jnp.zeros(4),
    "b_matchup"               : jnp.zeros((4, 4)),
    "b_last3_off_epa_avg"     : jnp.array(0.0),
    "b_last3_def_epa_avg"     : jnp.array(0.0),
    "b_last3_pts_scored_avg"  : jnp.array(0.0),
    "b_last3_pts_allowed_avg" : jnp.array(0.0),
    "b_days_since_last_game"  : jnp.array(0.0),
    "b_close_play_count_delta": jnp.array(0.0),
    "b_away_elevation_delta"  : jnp.array(0.0),
}

init_e = {**base_init}
init_f = {**base_init,
          "sigma_attack" : jnp.array(0.05),
          "alpha_ts_raw" : jnp.zeros(N_team_seasons),
          "sigma_defense": jnp.array(0.05),
          "delta_ts_raw" : jnp.zeros(N_team_seasons)}


# ---------------------------------------------------------------------------
# Fit function
# ---------------------------------------------------------------------------
def run_diagnostic(label, model_fn, init_vals, model_kwargs):
    print(f"  Fitting Model {label}...", end=" ", flush=True)
    nuts  = NUTS(model_fn, target_accept_prob=0.9,
                 init_strategy=init_to_value(values=init_vals))
    mcmc_ = MCMC(nuts, num_warmup=1000, num_samples=1000,
                 num_chains=1, progress_bar=False)
    numpyro.enable_validation(False)
    try:
        t0 = time.time()
        mcmc_.run(random.PRNGKey(0), **model_kwargs)
        elapsed = time.time() - t0
    finally:
        numpyro.enable_validation(True)
    extra_ = mcmc_.get_extra_fields()
    n_div  = int(extra_['diverging'].sum()) if 'diverging' in extra_ else 0
    samp_  = mcmc_.get_samples()
    idata_ = az.from_numpyro(mcmc_)
    print(f"done ({elapsed:.0f}s)  divergences={n_div}")
    return samp_, idata_, n_div, elapsed


print("=" * 60)
print("Running diagnostic fits (1 chain, 1000 warmup, 1000 samples)")
print("=" * 60)

samp_e, idata_e, ndiv_e, t_e = run_diagnostic(
    "E (features only)", model_e, init_e,
    {"data": training_data, "N_teams": N_teams,
     "log_points": log_points,
     "opp_def_archetype_idx": opp_def_archetype_idx}
)
samp_f, idata_f, ndiv_f, t_f = run_diagnostic(
    "F (team-season intercepts)", model_f, init_f,
    {"data": training_data, "N_teams": N_teams,
     "log_points": log_points,
     "opp_def_archetype_idx": opp_def_archetype_idx,
     "team_season_idx": team_season_idx,
     "opp_season_idx": opp_season_idx,
     "N_team_seasons": N_team_seasons}
)
print()


# ---------------------------------------------------------------------------
# ESS
# ---------------------------------------------------------------------------
def get_min_ess(idata_, extra_vars=None):
    scalar_vars = ["mu_league", "hfa_league", "sigma_conference",
                   "sigma_hfa_team", "sp_weight", "b_close_game_epa"]
    if extra_vars:
        scalar_vars += extra_vars
    ess = az.ess(idata_, var_names=scalar_vars)
    return min(float(ess[v].values.min())
               for v in scalar_vars if v in ess)

ess_e = get_min_ess(idata_e)
ess_f = get_min_ess(idata_f, extra_vars=["sigma_attack", "sigma_defense"])


# ---------------------------------------------------------------------------
# Log-variance check — 100 draws
# ---------------------------------------------------------------------------
def logvar_check(samp_, has_ts=False):
    mu_l  = np.array(samp_['mu_league'])
    hfa_l = np.array(samp_['hfa_league'])
    mu_c  = np.array(samp_['mu_conference'])
    hfa   = np.array(samp_['hfa_team'])
    sig   = np.array(samp_['sigma_obs'])
    bm    = np.array(samp_['b_matchup'])
    if has_ts:
        alp = np.array(samp_['alpha_ts'])   # (S, 393)
        dlt = np.array(samp_['delta_ts'])
    rng   = np.random.default_rng(0)
    draws = rng.choice(len(mu_l), size=100, replace=False)
    conf_draws = {c: [] for c in CONFERENCES}
    for d in draws:
        log_mu = (
            mu_l[d]
            + mu_c[d, conf_idx_arr]
            + hfa_l[d] * is_home_arr
            + hfa[d, team_idx_arr] * is_home_arr
            + bm[d, off_arch_arr, opp_def_arch_arr]
        )
        if has_ts:
            log_mu = log_mu + alp[d, ts_idx_arr] - dlt[d, os_idx_arr]
        sig_o  = sig[d, conf_idx_arr]
        y_log  = rng.normal(loc=log_mu, scale=sig_o)
        for c in CONFERENCES:
            mask = conf_labels == c
            conf_draws[c].append(y_log[mask].var())
    n_pass = 0
    for c in CONFERENCES:
        dc = np.array(conf_draws[c])
        lo, hi = np.percentile(dc, 5), np.percentile(dc, 95)
        if lo <= obs_log_var[c] <= hi:
            n_pass += 1
    return n_pass

print("Running log-variance checks...", flush=True)
vpass_e = logvar_check(samp_e, has_ts=False)
vpass_f = logvar_check(samp_f, has_ts=True)
print("done.")
print()


# ---------------------------------------------------------------------------
# Mean checks
# ---------------------------------------------------------------------------
def mean_checks(samp_, has_ts=False):
    mu_l  = np.array(samp_['mu_league']).mean()
    hfa_l = np.array(samp_['hfa_league']).mean()
    mu_c  = np.array(samp_['mu_conference']).mean(axis=0)
    hfa   = np.array(samp_['hfa_team']).mean(axis=0)
    sig   = np.array(samp_['sigma_obs']).mean(axis=0)
    bm    = np.array(samp_['b_matchup']).mean(axis=0)
    if has_ts:
        alp = np.array(samp_['alpha_ts']).mean(axis=0)
        dlt = np.array(samp_['delta_ts']).mean(axis=0)

    log_mu = (
        mu_l
        + mu_c[conf_idx_arr]
        + hfa_l * is_home_arr
        + hfa[team_idx_arr] * is_home_arr
        + bm[off_arch_arr, opp_def_arch_arr]
    )
    if has_ts:
        log_mu = log_mu + alp[ts_idx_arr] - dlt[os_idx_arr]

    sig_o   = sig[conf_idx_arr]
    mu_pred = np.exp(log_mu + 0.5 * sig_o**2) - 1.0

    tmp = df[['team_name', 'conference', 'points_scored',
              'season']].copy()
    tmp['mu_pred'] = mu_pred

    team_summ = (tmp.groupby('team_name')
                 .agg(obs=('points_scored', 'mean'),
                      pred=('mu_pred', 'mean'))
                 .assign(diff=lambda x: x['pred'] - x['obs']))
    conf_summ = (tmp.groupby('conference')
                 .agg(obs=('points_scored', 'mean'),
                      pred=('mu_pred', 'mean'))
                 .assign(diff=lambda x: x['pred'] - x['obs']))

    # Team-season level check for Model F
    if has_ts:
        ts_summ = (tmp.groupby(['team_name', 'season'])
                   .agg(obs=('points_scored', 'mean'),
                        pred=('mu_pred', 'mean'))
                   .assign(diff=lambda x: x['pred'] - x['obs']))
        n_ts_pass = int((ts_summ['diff'].abs() <= 2.0).sum())
        wt_ts     = float(ts_summ['diff'].abs().max())
    else:
        n_ts_pass = None
        wt_ts     = None

    n_team_pass = int((team_summ['diff'].abs() <= 2.0).sum())
    n_conf_pass = int((conf_summ['diff'].abs() <= 3.0).sum())
    wt          = float(team_summ['diff'].abs().max())
    wc          = float(conf_summ['diff'].abs().max())
    return n_team_pass, n_conf_pass, wt, wc, n_ts_pass, wt_ts, team_summ, conf_summ

tp_e, cp_e, wt_e, wc_e, _, _, tsumm_e, csumm_e = mean_checks(samp_e, has_ts=False)
tp_f, cp_f, wt_f, wc_f, ts_f, wts_f, tsumm_f, csumm_f = mean_checks(samp_f, has_ts=True)


# ---------------------------------------------------------------------------
# sigma_attack for Model F
# ---------------------------------------------------------------------------
sa_f = np.array(samp_f['sigma_attack'])
sd_f = np.array(samp_f['sigma_defense'])


# ---------------------------------------------------------------------------
# Top 10 / bottom 10 team detail
# ---------------------------------------------------------------------------
team_obs_mean  = (df.groupby('team_name')['points_scored']
                  .mean().sort_values(ascending=False))
top10_teams    = team_obs_mean.head(10).index.tolist()
bottom10_teams = team_obs_mean.tail(10).index.tolist()

def print_team_detail(label, tsumm, samp_, has_ts):
    print(f"\n  Model {label} — top 10 and bottom 10 teams:")
    print(f"  {'Team':<28s}  {'Obs':>7s}  {'Pred':>7s}  {'Diff':>7s}")
    print("  " + "-" * 52)
    for team in top10_teams:
        row = tsumm.loc[team]
        print(f"  {team:<28s}  {row['obs']:>7.2f}  "
              f"{row['pred']:>7.2f}  {row['diff']:>+7.2f}")
    print("  ...")
    for team in bottom10_teams:
        row = tsumm.loc[team]
        print(f"  {team:<28s}  {row['obs']:>7.2f}  "
              f"{row['pred']:>7.2f}  {row['diff']:>+7.2f}")

print_team_detail("E", tsumm_e, samp_e, has_ts=False)
print_team_detail("F", tsumm_f, samp_f, has_ts=True)
print()


# ---------------------------------------------------------------------------
# Conference detail — both models
# ---------------------------------------------------------------------------
print("=== Conference-level mean check ===")
print(f"  {'Conference':<22s}  {'Obs':>7s}  "
      f"{'E Pred':>7s}  {'E Diff':>7s}  {'E Status':>8s}  "
      f"{'F Pred':>7s}  {'F Diff':>7s}  {'F Status':>8s}")
print("  " + "-" * 82)
for c_name in CONFERENCES:
    re = csumm_e.loc[c_name]
    rf = csumm_f.loc[c_name]
    se = "PASS" if abs(re['diff']) <= 3.0 else "FLAG"
    sf = "PASS" if abs(rf['diff']) <= 3.0 else "FLAG"
    print(f"  {c_name:<22s}  {re['obs']:>7.2f}  "
          f"{re['pred']:>7.2f}  {re['diff']:>+7.2f}  {se:>8s}  "
          f"{rf['pred']:>7.2f}  {rf['diff']:>+7.2f}  {sf:>8s}")
print()


# ---------------------------------------------------------------------------
# sigma_obs per conference — both models
# ---------------------------------------------------------------------------
print("=== sigma_obs posterior mean per conference ===")
sig_e = np.array(samp_e['sigma_obs']).mean(axis=0)
sig_f = np.array(samp_f['sigma_obs']).mean(axis=0)
print(f"  {'Conference':<22s}  {'Obs log-var':>11s}  "
      f"{'E sigma':>8s}  {'E sig²':>8s}  {'E gap':>7s}  "
      f"{'F sigma':>8s}  {'F sig²':>8s}  {'F gap':>7s}")
print("  " + "-" * 88)
for i, c in enumerate(CONFERENCES):
    olv = obs_log_var[c]
    se  = float(sig_e[i]);  sf = float(sig_f[i])
    print(f"  {c:<22s}  {olv:>11.4f}  "
          f"{se:>8.4f}  {se**2:>8.4f}  {olv-se**2:>+7.4f}  "
          f"{sf:>8.4f}  {sf**2:>8.4f}  {olv-sf**2:>+7.4f}")
print()


# ---------------------------------------------------------------------------
# Summary table
# ---------------------------------------------------------------------------
print("=" * 66)
print("MODEL E vs MODEL F — DIAGNOSTIC SUMMARY")
print("(1 chain, 1000 warmup, 1000 samples)")
print("=" * 66)
print(f"  {'Metric':<40s}  {'Model E':>10s}  {'Model F':>10s}")
print("  " + "-" * 60)
print(f"  {'Description':<40s}  {'Feat only':>10s}  {'Team-season':>10s}")
print(f"  {'Fit time (s)':<40s}  {t_e:>10.0f}  {t_f:>10.0f}")
print(f"  {'Divergences':<40s}  {ndiv_e:>10d}  {ndiv_f:>10d}")
print(f"  {'Min ESS_bulk':<40s}  {ess_e:>10.0f}  {ess_f:>10.0f}")
print(f"  {'sigma_attack mean':<40s}  {'N/A':>10s}  {sa_f.mean():>10.4f}")
print(f"  {'sigma_defense mean':<40s}  {'N/A':>10s}  {sd_f.mean():>10.4f}")
print(f"  {'Log-var check: passed / 10':<40s}  {vpass_e:>10d}  {vpass_f:>10d}")
print(f"  {'Team mean check: passed / 131':<40s}  {tp_e:>10d}  {tp_f:>10d}")
print(f"  {'Team mean check: worst gap (pts)':<40s}  {wt_e:>10.2f}  {wt_f:>10.2f}")
print(f"  {'Conf mean check: passed / 10':<40s}  {cp_e:>10d}  {cp_f:>10d}")
print(f"  {'Conf mean check: worst gap (pts)':<40s}  {wc_e:>10.2f}  {wc_f:>10.2f}")
if ts_f is not None:
    print(f"  {'Team-season check: passed / 393':<40s}  {'N/A':>10s}  {ts_f:>10d}")
    print(f"  {'Team-season check: worst gap (pts)':<40s}  {'N/A':>10s}  {wts_f:>10.2f}")
print("=" * 66)
print()
print("Cell 14 complete.")

Team-season index built.
  Seasons          : [np.int64(2022), np.int64(2023), np.int64(2024)]
  N_team_seasons   : 384  (expect 131 x 3 = 393)

  team_season_idx  : shape=(3214,)  range=[0, 383]
  opp_season_idx   : shape=(3214,)  range=[0, 383]

Running diagnostic fits (1 chain, 1000 warmup, 1000 samples)
  Fitting Model E (features only)... done (88s)  divergences=0
  Fitting Model F (team-season intercepts)... done (93s)  divergences=0

Running log-variance checks...
done.


  Model E — top 10 and bottom 10 teams:
  Team                              Obs     Pred     Diff
  ----------------------------------------------------
  SMU                             39.69    27.39   -12.30
  Oregon                          38.59    24.09   -14.50
  UTSA                            38.20    27.57   -10.63
  Memphis                         36.33    28.70    -7.63
  USC                             35.43    24.41   -11.02
  Jacksonville State              35.41    25.33   -10.08
  Ohio State   